# TradeSense · TGNN branch · Stage B2: download and audit of the Aug–Oct 2024 free days → `12_tgnn_tardis_stageB2_download_audit_aug_oct_v1`

**Where we are.** Six free first-of-month sets (BTCUSDT and ETHUSDT, 2024-05-01, 2024-06-01, 2024-07-01) are already downloaded and audited. TGNN Protocol Design v2 (approved in direction) assigns them to **train (May, June)** and **validation (July)**, and reserves **August, September and October 2024** as the **test months**. Stage B1 (a headers-only check you ran) found all 18 Aug–Oct files **reachable**; reachable does **not** mean complete, free to use or covered by the licence.

**What this notebook does (and nothing more).** For the six new sets it (1) re-checks the headers (no download), (2) downloads them after the safety gate, and (3) runs the *same* audit as notebooks 07/10 and the *same* walk-the-book test as notebooks 09/10:

| symbol | days |
|---|---|
| BTCUSDT | 2024-08-01, 2024-09-01, 2024-10-01 |
| ETHUSDT | 2024-08-01, 2024-09-01, 2024-10-01 |

Reported sizes from Stage B1: about 1.54 GB compressed in total (207–309 MB per set). My estimate of the run time is **45–75 minutes**; this is extrapolated from the earlier runs, not measured.

**Test-month discipline (protocol v2).** The audit and walk-the-book test only describe data quality; **no model is trained or scored here** and nothing from these days may be used to choose model settings. The numbers printed here are audit evidence, not model-selection evidence. If you ever see a result here that tempts a design change in the TGNN, tell Claude and it will be logged as a protocol deviation instead of acted on silently.

**Nothing else.** No other dates or symbols, no graph tensors, no training, no synthetic events, no fusion. Existing folders are never touched: a set whose output folder already exists is **skipped**, never overwritten. Each set gets its own versioned folder `TGNN_Tardis_Binance_Spot_<SYMBOL>_<DAY>_v1` (and `TGNN_Tardis_WalkBook_<SYMBOL>_<DAY>_v1`), created by renaming a hidden staging folder only if that set's execution (E) and dataset-integrity (D) tests pass. The reconstruction assessment (R) never decides whether files are saved.

**Safety switch.** Cell 3 refuses to download until you set `I_HAVE_READ_THE_TERMS = True` in cell 1. Terms: internal research/educational use; **no redistribution of raw data, per-trade tables derived from it, or trained models**; publication of results is not addressed in the terms — ask Tardis in writing.

**Colab may disconnect.** Sets already saved are skipped on the next Run all, so re-running resumes. If a set fails, the others continue and the status table says what happened.

**Nothing here validates the reconstructed book scientifically or detects manipulation.** "Consistent" means consistent with the provider's own snapshots of the same feed.

## 1 · Settings (the only cell you may need to edit)

In [ ]:
import os, gc, json, time, shutil, traceback, re
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
BASE_URL = os.environ.get('TARDIS_BASE_URL', 'https://datasets.tardis.dev')       # env var only used by my mock-server tests
EXCHANGE = 'binance'
JOBS = [('BTCUSDT', '2024-08-01'), ('ETHUSDT', '2024-08-01'), ('BTCUSDT', '2024-09-01'), ('ETHUSDT', '2024-09-01'), ('BTCUSDT', '2024-10-01'), ('ETHUSDT', '2024-10-01')]      # the six test-month sets (Stage B2)
PREV_SETS = [('BTCUSDT', '2024-05-01'), ('ETHUSDT', '2024-05-01'), ('BTCUSDT', '2024-06-01'), ('ETHUSDT', '2024-06-01'), ('BTCUSDT', '2024-07-01'), ('ETHUSDT', '2024-07-01')]      # already saved earlier; only READ for the final table
I_HAVE_READ_THE_TERMS = True      # <-- change to True ONLY after you have read the Tardis Terms of Service: https://docs.tardis.dev/legal/terms-of-service
INCLUDE_BOOK_SNAPSHOT_25 = True    # needed for the reconstruction criteria
MAX_GB_PER_SET = 8.0               # safety limit for the three files of one set
ALLOW_PARTIAL = False              # False = stop if ANY file is unavailable; True = skip unavailable sets and continue with the rest
RUN_WALK_THE_BOOK = True           # run notebook 09's test on each saved set
WORKDIR = Path('/content') if Path('/content').exists() else Path(os.getcwd())
def final_dir(sym, day): return BASE / f'TGNN_Tardis_Binance_Spot_{sym}_{day}_v1'
def walk_dir(sym, day): return BASE / f'TGNN_Tardis_WalkBook_{sym}_{day}_v1'
print('sets to process:'); [print('  ', s, d, '-> already saved, will be skipped' if final_dir(s, d).exists() else '-> to do') for s, d in JOBS]

Mounted at /content/drive
sets to process:
   BTCUSDT 2024-08-01 -> already saved, will be skipped
   ETHUSDT 2024-08-01 -> already saved, will be skipped
   BTCUSDT 2024-09-01 -> already saved, will be skipped
   ETHUSDT 2024-09-01 -> to do
   BTCUSDT 2024-10-01 -> to do
   ETHUSDT 2024-10-01 -> to do


[None, None, None, None, None, None]

## 2 · The two analysis programs (written to the Colab working folder, not to Drive)

These two cells contain the **same code as notebooks 07 and 09** (with the symbol/day made adjustable and the changes listed above). They only save a text file each; nothing runs here. You do not need to read or edit them.

In [ ]:
%%writefile tardis_body_audit.py
import os, sys, io, re, gc, json, time, math, gzip, uuid, shutil, hashlib, heapq, warnings, resource
from array import array
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import requests
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 100)
pass      # the driver cell has already mounted Drive
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
assert BASE.is_dir(), f'Drive folder not found: {BASE}'
EXCHANGE = 'binance'; SYMBOL = JOB_SYMBOL; DAY = JOB_DAY
Y, M, D = DAY.split('-')
TYPES = {'incremental_book_L2': True, 'trades': True, 'book_snapshot_25': False}          # name -> required?
BASE_URL = os.environ.get('TARDIS_BASE_URL', 'https://datasets.tardis.dev')                # env var only used by my mock-server tests
OUT_FINAL = BASE / f'TGNN_Tardis_Binance_Spot_{SYMBOL}_{DAY}_v1'
STAGE = BASE / ('.tardis_stage_' + uuid.uuid4().hex[:8])
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); T_ALL = time.time(); STAGES = {}; REPORT = {}
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def peak_mem_gb(): return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6      # Linux: KB -> GB
def tick(name, t0): STAGES[name] = {'seconds': round(time.time() - t0, 1), 'peak_mem_gb_so_far': round(peak_mem_gb(), 2)}; print(f'[{name}] {STAGES[name]["seconds"]} s | peak memory so far {STAGES[name]["peak_mem_gb_so_far"]} GB')
# fingerprint (size + modified time, no hashing) of EVERYTHING already inside the research-datasets folder
t0 = time.time(); PIN0 = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file()}
print(f'fingerprinted {len(PIN0)} existing files (they will be re-checked at the end)'); tick('fingerprint', t0)
HEADERS = {'User-Agent': 'TradeSense-research-notebook/1.0 (final-year project)'}
API_KEY = None            # leave as None: the first day of each month needs no key. Only set one if you bought a subscription.
if API_KEY: HEADERS['Authorization'] = 'Bearer ' + API_KEY
def url_for(dt): return f'{BASE_URL}/v1/{EXCHANGE}/{dt}/{Y}/{M}/{D}/{SYMBOL}.csv.gz'
def fname_for(dt): return f'{EXCHANGE}_{dt}_{DAY}_{SYMBOL}.csv.gz'
for dt in TYPES: print(dt, '->', url_for(dt))

I_HAVE_READ_THE_TERMS = DRIVER_I_HAVE_READ_THE_TERMS
INCLUDE_BOOK_SNAPSHOT_25 = DRIVER_INCLUDE_S25
MAX_TOTAL_GB = DRIVER_MAX_GB

du = shutil.disk_usage(str(BASE)); free_gb = du.free / 1e9
loc = shutil.disk_usage('/content' if Path('/content').exists() else '/tmp')
print(f'Drive/target volume: total {du.total/1e9:.1f} GB | free {free_gb:.1f} GB   ||  local disk free {loc.free/1e9:.1f} GB')
REPORT['storage'] = {'target_total_gb': du.total / 1e9, 'target_free_gb': free_gb, 'local_free_gb': loc.free / 1e9}

def probe(url):
    # Plain GET with stream=True: only the response HEADERS are read, then the connection is closed (no file body is downloaded).
    # (A HEAD request can be answered differently from GET by some servers, so we do not rely on it.)
    out = {'url': url, 'status': None, 'bytes': None, 'content_type': None, 'last_modified': None, 'error': None, 'body': None}
    try:
        r = requests.get(url, headers=HEADERS, stream=True, allow_redirects=True, timeout=60); out['status'] = r.status_code
        out['content_type'] = r.headers.get('Content-Type'); out['last_modified'] = r.headers.get('Last-Modified')
        if r.status_code in (200, 206): out['bytes'] = int(r.headers['Content-Length']) if 'Content-Length' in r.headers else None
        else: out['body'] = r.raw.read(400, decode_content=True).decode('utf-8', 'replace')    # small error message from the server, if any
        r.close()
    except Exception as e: out['error'] = type(e).__name__ + ': ' + str(e)[:200]
    return out
PROBE = {dt: probe(url_for(dt)) for dt in TYPES if TYPES[dt] or INCLUDE_BOOK_SNAPSHOT_25}
display(pd.DataFrame(PROBE).T[['status', 'bytes', 'content_type', 'last_modified', 'error', 'body']])
REPORT['preflight'] = PROBE
known = [v['bytes'] for v in PROBE.values() if v['bytes']]; total_gb = sum(known) / 1e9
bad = {k: v for k, v in PROBE.items() if v['status'] not in (200, 206) or (v['bytes'] is not None and v['bytes'] < 1000)}    # tiny file = Tardis' empty-gzip answer for 'no data'
print(f'\nTotal compressed size of the files above: {total_gb:.2f} GB (files with unknown size: {sum(1 for v in PROBE.values() if not v["bytes"])})')
if bad:
    print('\n!! The server did not return the file for:', {k: (v['status'], v['error'], v['body']) for k, v in bad.items()})
    print('   URLs tried:'); [print('    ', v['url']) for v in bad.values()]
    try:    # read-only metadata call (no dataset download): what does Tardis itself list for this exchange/symbol?
        mj = requests.get(f'https://api.tardis.dev/v1/exchanges/{EXCHANGE}', timeout=45).json(); ds = mj.get('datasets') or {}
        sy = [x for x in ds.get('symbols', []) if str(x.get('id', '')).upper() == SYMBOL]
        print('   Tardis metadata -> supportsDatasets:', mj.get('supportsDatasets'), '| exportedUntil:', ds.get('exportedUntil'), '| symbol entry:', sy[:1] if sy else 'NOT LISTED in datasets.symbols')
        REPORT['tardis_metadata_check'] = {'supportsDatasets': mj.get('supportsDatasets'), 'exportedUntil': ds.get('exportedUntil'), 'symbol_entry': sy[:1]}
    except Exception as e: print('   (metadata check failed:', type(e).__name__, str(e)[:120], ')')
    print('   404 = file does not exist for that date/type; 401/403 = free access does not apply to this file. Do NOT try to work around this; paste this output to Claude.')
    raise RuntimeError('STOPPED ON PURPOSE: availability check failed (see message above). Nothing was downloaded.')
if total_gb > MAX_TOTAL_GB: raise RuntimeError(f'STOPPED ON PURPOSE: files total {total_gb:.1f} GB > MAX_TOTAL_GB={MAX_TOTAL_GB}. Raise the limit only if you have the space.')
if total_gb * 1.2 + 2 > free_gb: raise RuntimeError(f'STOPPED ON PURPOSE: not enough free space ({free_gb:.1f} GB free, need about {total_gb*1.2+2:.1f} GB).')
if not I_HAVE_READ_THE_TERMS:
    print('\n' + '=' * 90 + '\nGATE: availability OK and space OK. NOTHING HAS BEEN DOWNLOADED.\nTo continue: read the Tardis Terms of Service (link in the first cell), set I_HAVE_READ_THE_TERMS = True above, then Run all.\n' + '=' * 90)
    raise RuntimeError('STOPPED ON PURPOSE at the safety gate (this is expected the first time).')
print('Gate passed. Proceeding to download.')

def download(url, dest, expected_bytes, tries=4):
    part = dest.with_name(dest.name + '.part'); t0 = time.time()
    for attempt in range(1, tries + 1):
        have = part.stat().st_size if part.exists() else 0
        hdr = dict(HEADERS)
        if have: hdr['Range'] = f'bytes={have}-'
        try:
            with requests.get(url, headers=hdr, stream=True, allow_redirects=True, timeout=(30, 120)) as r:
                if r.status_code == 416 and expected_bytes and have == expected_bytes: break
                if r.status_code not in (200, 206): raise RuntimeError(f'HTTP {r.status_code}')
                mode = 'ab' if (r.status_code == 206 and have) else 'wb'; got = have if mode == 'ab' else 0; nxt = 0.05
                with open(part, mode) as f:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        f.write(chunk); got += len(chunk)
                        if expected_bytes and got / expected_bytes >= nxt: print(f'   {dest.name}: {got/1e6:,.0f} MB ({got/expected_bytes:.0%}) {time.time()-t0:.0f}s'); nxt += 0.05
            break
        except Exception as e:
            print(f'   attempt {attempt} failed: {type(e).__name__}: {str(e)[:120]}'); time.sleep(3 * attempt)
            if attempt == tries: raise
    size = part.stat().st_size
    if expected_bytes and size != expected_bytes: raise RuntimeError(f'size mismatch for {dest.name}: got {size}, server said {expected_bytes}')
    part.rename(dest); return size
RAW = STAGE / 'raw'; RAW.mkdir(parents=True, exist_ok=False); DL = {}
t0 = time.time()
for dt, req in TYPES.items():
    if not (req or INCLUDE_BOOK_SNAPSHOT_25): continue
    dest = RAW / fname_for(dt); print('downloading', dt); size = download(url_for(dt), dest, PROBE[dt]['bytes'])
    DL[dt] = {'file': dest.name, 'bytes': size, 'url': url_for(dt)}
tick('download', t0)
for dt in DL: print(dt, f"{DL[dt]['bytes']/1e6:,.1f} MB"); DL[dt]['path'] = RAW / DL[dt]['file']
REPORT['download'] = {k: {x: y for x, y in v.items() if x != 'path'} for k, v in DL.items()}

t0 = time.time()
for dt, d in DL.items():
    p = d['path']; d['sha256'] = sha256_file(p); n_lines = 0; n_bytes = 0; ok = True
    try:
        with gzip.open(p, 'rb') as f:
            while True:
                b = f.read(32 << 20)
                if not b: break
                n_bytes += len(b); n_lines += b.count(b'\n')
    except Exception as e: ok = False; d['gzip_error'] = str(e)[:200]
    d.update({'gzip_ok': ok, 'uncompressed_bytes': n_bytes, 'lines_incl_header': n_lines})
    print(f"{dt}: gzip_ok={ok} | uncompressed {n_bytes/1e9:.2f} GB | lines {n_lines:,} | sha256 {d['sha256'][:16]}...")
tick('integrity', t0); REPORT['integrity'] = {k: {x: y for x, y in v.items() if x != 'path'} for k, v in DL.items()}

EXPECTED = {
 'incremental_book_L2': ['exchange', 'symbol', 'timestamp', 'local_timestamp', 'is_snapshot', 'side', 'price', 'amount'],
 'trades': ['exchange', 'symbol', 'timestamp', 'local_timestamp', 'id', 'side', 'price', 'amount'],
 'book_snapshot_25': ['exchange', 'symbol', 'timestamp', 'local_timestamp'] + [f'{s}[{i}].{k}' for i in range(25) for s in ('asks', 'bids') for k in ('price', 'amount')]}
def col_types(dt):
    cols = EXPECTED[dt]; t = {'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64()}
    if dt == 'incremental_book_L2': t.update({'is_snapshot': pa.bool_(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()})
    elif dt == 'trades': t.update({'id': pa.string(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()})
    else: t.update({c: pa.float64() for c in cols[4:]})
    return t
def reader(dt, block_mb=48):
    ro = pacsv.ReadOptions(block_size=block_mb << 20); co = pacsv.ConvertOptions(column_types=col_types(dt), strings_can_be_null=False, true_values=['true', 'True', 'TRUE'], false_values=['false', 'False', 'FALSE'])
    return pacsv.open_csv(DL[dt]['path'], read_options=ro, convert_options=co)
SCHEMA = {}
for dt, d in DL.items():
    with gzip.open(d['path'], 'rt') as f: header = f.readline().strip().split(','); first = f.readline().strip()
    match = header == EXPECTED[dt]; SCHEMA[dt] = {'columns_found': len(header), 'matches_documented_header': match, 'missing': [c for c in EXPECTED[dt] if c not in header], 'extra': [c for c in header if c not in EXPECTED[dt]]}
    print(f'{dt}: {len(header)} columns | matches documented header: {match} | missing {SCHEMA[dt]["missing"][:4]} extra {SCHEMA[dt]["extra"][:4]}')
    if dt != 'book_snapshot_25': print('   header:', ','.join(header))
REPORT['schema'] = SCHEMA
assert all(v['matches_documented_header'] for v in SCHEMA.values()), 'A file does not match the documented header: STOP and paste the output above to Claude (do not guess column meanings).'

def vc(col): return {x['values']: x['counts'] for x in pc.value_counts(col).to_pylist()}
DAY0_US = int(pd.Timestamp(DAY, tz='UTC').timestamp() * 1e6); DAY1_US = DAY0_US + 86_400_000_000
GAP_TH = {'>1s': 1e6, '>10s': 1e7, '>60s': 6e7, '>300s': 3e8}
def iso_us(v): return None if v is None else pd.Timestamp(int(v), unit='us', tz='UTC').isoformat()
class Gaps:
    def __init__(s): s.last = None; s.cnt = Counter(); s.top = []
    def add(s, lts):                        # lts: distinct-or-repeated increasing-ish local timestamps (int64)
        x = lts if s.last is None else np.r_[s.last, lts]; d = np.diff(x); s.last = int(lts[-1])
        for k, th in GAP_TH.items(): s.cnt[k] += int((d > th).sum())
        for i in np.flatnonzero(d > 6e7)[:40]: s.top.append((int(d[i]), int(x[i]))); s.top = sorted(s.top, reverse=True)[:10]
def scan_incremental():
    rd = reader('incremental_book_L2'); n = 0; tmin = tmax = lmin = lmax = None; inv_t = inv_l = 0; last_t = last_l = None; snap_rows = 0; pre_rows = 0; seen_snap = False
    side_ct = Counter(); zero_amt = 0; neg = 0; nan = 0; badpx = 0; gaps = Gaps(); batches = []; cur = None; prev_flag = None; hours = Counter(); out_day = 0; lat_b = []; prev_l_for_gap = None; sym_ct = Counter(); ex_ct = Counter()
    for b in rd:
        ts = b.column('timestamp').to_numpy(); ls = b.column('local_timestamp').to_numpy(); fl = b.column('is_snapshot').to_numpy(zero_copy_only=False).astype(bool); amt = b.column('amount').to_numpy(); px = b.column('price').to_numpy()
        isb = pc.equal(b.column('side'), 'bid').to_numpy(zero_copy_only=False).astype(bool)
        m = len(ts); base = n; n += m
        tmin = int(ts.min()) if tmin is None else min(tmin, int(ts.min())); tmax = int(ts.max()) if tmax is None else max(tmax, int(ts.max())); lmin = int(ls.min()) if lmin is None else min(lmin, int(ls.min())); lmax = int(ls.max()) if lmax is None else max(lmax, int(ls.max()))
        xt = ts if last_t is None else np.r_[last_t, ts]; inv_t += int((np.diff(xt) < 0).sum()); last_t = int(ts[-1]); xl = ls if last_l is None else np.r_[last_l, ls]; inv_l += int((np.diff(xl) < 0).sum()); last_l = int(ls[-1])
        for k, v in vc(b.column('side')).items(): side_ct[k] += v
        for k, v in vc(b.column('symbol')).items(): sym_ct[k] += v
        for k, v in vc(b.column('exchange')).items(): ex_ct[k] += v
        lat_b.append((ls - ts)[:: max(1, len(ts) // 20000)]); out_day += int(((ls < DAY0_US) | (ls >= DAY1_US)).sum())
        zero_amt += int((amt == 0).sum()); neg += int((amt < 0).sum()); nan += int(np.isnan(amt).sum() + np.isnan(px).sum()); badpx += int((px <= 0).sum())
        snap_rows += int(fl.sum())
        if not seen_snap:
            if fl.any(): pre_rows += int(np.argmax(fl)); seen_snap = True
            else: pre_rows += m
        chg = np.flatnonzero(fl[1:] != fl[:-1]) + 1 if m > 1 else np.array([], int); starts = [0] + [int(x) for x in chg]; bounds = starts + [m]
        for j_, s0 in enumerate(starts):
            e0 = bounds[j_ + 1]; seg_flag = bool(fl[s0]); cont = (j_ == 0 and prev_flag is not None and seg_flag == prev_flag)
            if seg_flag:
                if cont and cur is not None: cur['rows'] += e0 - s0; cur['bid_rows'] += int(isb[s0:e0].sum()); cur['ask_rows'] += int((~isb[s0:e0]).sum())
                else:
                    if cur is not None: batches.append(cur)
                    cur = {'start_row': base + s0, 'local_ts': int(ls[s0]), 'rows': e0 - s0, 'bid_rows': int(isb[s0:e0].sum()), 'ask_rows': int((~isb[s0:e0]).sum())}
            elif cur is not None: batches.append(cur); cur = None
        prev_flag = bool(fl[-1])
        hh = pd.Series((ls - DAY0_US) // 3_600_000_000).value_counts()
        for k, v in hh.items(): hours[int(k)] += int(v)
        gaps.add(ls)
        if n % 20_000_000 < m: print(f'   scanned {n:,} rows')
    if cur is not None: batches.append(cur)
    lat_b = np.concatenate(lat_b) if lat_b else np.array([0])
    return {'local_outside_day_rows': out_day, 'book_local_minus_exchange_us_median': float(np.median(lat_b)), 'book_local_minus_exchange_us_p99': float(np.percentile(lat_b, 99)), 'rows': n, 'timestamp_min': iso_us(tmin), 'timestamp_max': iso_us(tmax), 'local_min': iso_us(lmin), 'local_max': iso_us(lmax), 'ts_inversions': inv_t, 'local_inversions': inv_l, 'snapshot_rows': snap_rows, 'rows_before_first_snapshot': pre_rows, 'sides': dict(side_ct), 'symbols': dict(sym_ct), 'exchanges': dict(ex_ct),
            'amount_zero_rows': zero_amt, 'amount_negative': neg, 'nan_cells': nan, 'price_le_zero': badpx, 'snapshot_batches': batches, 'rows_per_utc_hour': {int(k): int(v) for k, v in sorted(hours.items())}, 'gap_counts': dict(gaps.cnt), 'largest_gaps_s': [(round(g / 1e6, 1), iso_us(t)) for g, t in gaps.top], 'timestamp_unit_check_ok': bool(tmin and 1.0e15 < tmin < 2.0e15)}
t0 = time.time(); SC = scan_incremental(); tick('scan_incremental', t0); REPORT['book_scan'] = SC
print(json.dumps({k: v for k, v in SC.items() if k not in ('snapshot_batches', 'rows_per_utc_hour')}, indent=1, default=jsonable))
print('snapshot batches:', len(SC['snapshot_batches'])); display(pd.DataFrame(SC['snapshot_batches']).assign(time_utc=lambda d: d.local_ts.map(iso_us)).head(30))
print('rows per UTC hour:', SC['rows_per_utc_hour'])

def scan_trades():
    rd = reader('trades'); n = 0; tmin = tmax = lmin = lmax = None; inv = 0; last = None; side_ct = Counter(); ids_first = ids_last = None; id_num = True; px_min, px_max, q_min, q_max = 1e18, -1e18, 1e18, -1e18; lat = []; out_day = 0; gaps = Gaps(); id_gap = 0; id_dup = 0; prev_id = None
    for b in rd:
        ts = b.column('timestamp').to_numpy(); ls = b.column('local_timestamp').to_numpy(); px = b.column('price').to_numpy(); q = b.column('amount').to_numpy(); n += len(ts)
        tmin = int(ts.min()) if tmin is None else min(tmin, int(ts.min())); tmax = int(ts.max()) if tmax is None else max(tmax, int(ts.max())); lmin = int(ls.min()) if lmin is None else min(lmin, int(ls.min())); lmax = int(ls.max()) if lmax is None else max(lmax, int(ls.max()))
        x = ls if last is None else np.r_[last, ls]; inv += int((np.diff(x) < 0).sum()); last = int(ls[-1]); px_min, px_max, q_min, q_max = min(px_min, float(px.min())), max(px_max, float(px.max())), min(q_min, float(q.min())), max(q_max, float(q.max()))
        for k_, v_ in vc(b.column('side')).items(): side_ct[k_] += v_
        lat.append((ls - ts)[:: max(1, len(ts) // 20000)]); gaps.add(ls); out_day += int(((ls < DAY0_US) | (ls >= DAY1_US)).sum())
        try:
            ids = pc.cast(b.column('id'), pa.int64()).to_numpy(zero_copy_only=False); xi = ids if prev_id is None else np.r_[prev_id, ids]; d = np.diff(xi); id_gap += int((d > 1).sum()); id_dup += int((d <= 0).sum()); prev_id = int(ids[-1])
        except Exception: id_num = False
    lat = np.concatenate(lat) if lat else np.array([0])
    return {'local_outside_day_rows': out_day, 'amount_min': q_min, 'rows': n, 'timestamp_min': iso_us(tmin), 'timestamp_max': iso_us(tmax), 'local_min': iso_us(lmin), 'local_max': iso_us(lmax), 'local_inversions': inv, 'sides': dict(side_ct), 'price_range': [px_min, px_max], 'amount_range': [q_min, q_max], 'id_numeric': id_num, 'id_gaps(>1)': id_gap, 'id_dup_or_backwards': id_dup,
            'local_minus_exchange_us_median': float(np.median(lat)), 'local_minus_exchange_us_p99': float(np.percentile(lat, 99)), 'gap_counts': dict(gaps.cnt), 'largest_gaps_s': [(round(g / 1e6, 1), iso_us(t)) for g, t in gaps.top]}
t0 = time.time(); TR = scan_trades(); tick('scan_trades', t0); REPORT['trades_scan'] = TR; print(json.dumps(TR, indent=1, default=jsonable))

class Book:
    def __init__(s, K): s.bids = {}; s.asks = {}; s.bh = []; s.ah = []; s.K = K
    def reset(s): s.bids.clear(); s.asks.clear(); s.bh.clear(); s.ah.clear()
    def upd(s, is_bid, p, a):
        if is_bid:
            d = s.bids
            if a > 0:
                if p not in d: heapq.heappush(s.bh, -p)
                d[p] = a
            else: d.pop(p, None)
        else:
            d = s.asks
            if a > 0:
                if p not in d: heapq.heappush(s.ah, p)
                d[p] = a
            else: d.pop(p, None)
    def best(s):
        bh = s.bh; bd = s.bids
        while bh and -bh[0] not in bd: heapq.heappop(bh)
        ah = s.ah; ad = s.asks
        while ah and ah[0] not in ad: heapq.heappop(ah)
        if len(bh) > 4 * len(bd) + 4000: s.bh = bh = [-p for p in bd]; heapq.heapify(bh)
        if len(ah) > 4 * len(ad) + 4000: s.ah = ah = list(ad); heapq.heapify(ah)
        return (-bh[0] if bh else float('nan')), (ah[0] if ah else float('nan'))
    def prune(s):
        K = s.K
        if K and len(s.bids) > K + 50:
            for p in sorted(s.bids)[:len(s.bids) - K]: del s.bids[p]
        if K and len(s.asks) > K + 50:
            for p in sorted(s.asks)[K:]: del s.asks[p]
    def top(s, n=25):
        b = heapq.nlargest(n, s.bids.items()); a = heapq.nsmallest(n, s.asks.items()); return b, a
    def fingerprint(s): return hashlib.sha256(json.dumps([sorted(s.bids.items())[-50:], sorted(s.asks.items())[:50], len(s.bids), len(s.asks)]).encode()).hexdigest()[:16]

S25_META = {}
def load_snapshot25_sample(stride=2000):
    """sample every `stride`-th row of Tardis' own book_snapshot_25 (used only as a CROSS-CHECK, keyed by local_timestamp)"""
    if 'book_snapshot_25' not in DL: return {}, 0
    rd = reader('book_snapshot_25', block_mb=32); samp = {}; n = 0; S25_META['rows_sampled'] = 0
    for b in rd:
        m = b.num_rows; idx = [i for i in range(m) if (n + i) % stride == 0]; n += m
        if not idx: continue
        sub = b.take(pa.array(idx)); L = sub.column('local_timestamp').to_numpy()
        ap = np.column_stack([np.asarray(sub.column(f'asks[{i}].price').to_numpy(zero_copy_only=False), float) for i in range(25)]); aa = np.column_stack([np.asarray(sub.column(f'asks[{i}].amount').to_numpy(zero_copy_only=False), float) for i in range(25)])
        bp = np.column_stack([np.asarray(sub.column(f'bids[{i}].price').to_numpy(zero_copy_only=False), float) for i in range(25)]); ba = np.column_stack([np.asarray(sub.column(f'bids[{i}].amount').to_numpy(zero_copy_only=False), float) for i in range(25)])
        S25_META['rows_sampled'] += len(L)
        for j in range(len(L)): samp[int(L[j])] = (ap[j], aa[j], bp[j], ba[j])
    return samp, n
TOPNS = (5, 10, 25)
def compare_top(book, snap):
    """compare the rebuilt book with ONE book_snapshot_25 row (both taken after the same completed message)"""
    ap, aa, bp, ba = snap; b, a = book.top(25); r = {}
    for N in TOPNS: r[N] = {'lv': 0, 'p': 0, 'q': 0, 'all_p': True, 'all_pq': True}
    for sp_, sa_, mine in ((bp, ba, b), (ap, aa, a)):
        for i in range(25):
            sp = sp_[i]
            if not np.isfinite(sp): continue
            okp = i < len(mine) and mine[i][0] == sp; okq = okp and math.isclose(mine[i][1], sa_[i], rel_tol=1e-9, abs_tol=1e-12)
            for N in TOPNS:
                if i < N:
                    d = r[N]; d['lv'] += 1; d['p'] += int(okp); d['q'] += int(okq)
                    if not okp: d['all_p'] = False
                    if not okq: d['all_pq'] = False
    bb_s = bp[0]; ba_s = ap[0]; bb_m = b[0][0] if b else float('nan'); ba_m = a[0][0] if a else float('nan')
    r['best_ok'] = bool(np.isfinite(bb_s) and np.isfinite(ba_s) and bb_m == bb_s and ba_m == ba_s)
    r['spread_diff_bps'] = float(abs((ba_m - bb_m) - (ba_s - bb_s)) / ((ba_s + bb_s) / 2) * 1e4) if np.isfinite(bb_s) and np.isfinite(ba_s) and np.isfinite(bb_m) and np.isfinite(ba_m) else float('nan')
    r['spread_equal'] = bool(r['spread_diff_bps'] == r['spread_diff_bps'] and r['spread_diff_bps'] < 1e-9)
    return r
def new_cmp(): return {'checkpoints': 0, 'excluded_crossed_or_locked': 0, 'lv': Counter(), 'p': Counter(), 'q': Counter(), 'all_p': Counter(), 'all_pq': Counter(), 'best_ok': 0, 'spread_equal': 0, 'spread_diff': [], 'by_hour': Counter(), 'by_hour_ok': Counter(), 'empty_book_skipped': 0}
def depth_stats(bk, mid):
    d10 = d25 = 0.0; far = 0; lo10, lo25, lo_far = mid * 0.999, mid * 0.9975, mid * 0.95; hi10, hi25, hi_far = mid * 1.001, mid * 1.0025, mid * 1.05
    for p, q in bk.bids.items():
        if p >= lo10: d10 += q
        if p >= lo25: d25 += q
        if p < lo_far: far += 1
    for p, q in bk.asks.items():
        if p <= hi10: d10 += q
        if p <= hi25: d25 += q
        if p > hi_far: far += 1
    return d10, d25, far
def tick_of(bk):
    ps = np.array(sorted(bk.bids)[-200:] + sorted(bk.asks)[:200]); d = np.diff(ps); d = np.round(d[d > 1e-9], 8)
    return float(d.min()) if len(d) else float('nan')
def run_replay(max_rows=None, run_unpruned=True, samp=None, collect=True, K=1000, verbose=True, run_pruned=True):
    rd = reader('incremental_book_L2'); variants = {'pruned1000': Book(K)} if run_pruned else {}
    if run_unpruned: variants['unpruned'] = Book(0)
    V = list(variants.items()); TL = {k: {'lts': array('q'), 'bb': array('d'), 'ba': array('d')} for k in variants}
    st = {'rows': 0, 'skipped_pre': 0, 'messages': 0, 'snap_batches': 0, 'checks': 0, 'locked': Counter(), 'crossed': Counter(), 'max_cross_run': Counter(), 'run': Counter(), 'resets': 0}
    cmp = {k: new_cmp() for k in variants}
    depth = {k: [] for k in variants}; next_sample = [None]; started = False; pL = None; pf = None; t0 = time.time(); done_rows = 0; snap_checks = []
    def end_msg(L, was_snap):
        st['messages'] += 1; st['checks'] += 1
        if was_snap: st['snap_batches'] += 1
        for k, bk in V:
            if bk.K: bk.prune()
            bb, ba = bk.best()
            if collect: tl = TL[k]; tl['lts'].append(L); tl['bb'].append(bb); tl['ba'].append(ba)
            if bb == bb and ba == ba:
                if bb > ba: st['crossed'][k] += 1; st['run'][k] += 1; st['max_cross_run'][k] = max(st['max_cross_run'][k], st['run'][k])
                elif bb == ba: st['locked'][k] += 1; st['run'][k] = 0
                else: st['run'][k] = 0
            if samp is not None and L in samp:
                c = cmp[k]
                if not (bb == bb and ba == ba): c['empty_book_skipped'] += 1
                elif bb >= ba: c['excluded_crossed_or_locked'] += 1       # Tardis removes overlapping levels from its snapshots, so a crossed/locked moment is not comparable
                else:
                    r = compare_top(bk, samp[L]); c['checkpoints'] += 1; hr = int((L - DAY0_US) // 3_600_000_000)
                    for N in TOPNS:
                        d = r[N]; c['lv'][N] += d['lv']; c['p'][N] += d['p']; c['q'][N] += d['q']; c['all_p'][N] += int(d['all_p']); c['all_pq'][N] += int(d['all_pq'])
                    c['best_ok'] += int(r['best_ok']); c['spread_equal'] += int(r['spread_equal']); c['spread_diff'].append(r['spread_diff_bps']); c['by_hour'][hr] += 1; c['by_hour_ok'][hr] += int(r[25]['all_pq'])
        if collect and (next_sample[0] is None or L >= next_sample[0]):
            next_sample[0] = L + 30_000_000
            for k, bk in V:
                bb, ba = bk.best()
                if bb == bb and ba == ba:
                    mid = (bb + ba) / 2; depth[k].append((L, mid, (ba - bb) / mid * 1e4, len(bk.bids), len(bk.asks)) + depth_stats(bk, mid))
    for b in rd:
        Ls = b.column('local_timestamp').to_numpy().tolist(); Fs = b.column('is_snapshot').to_numpy(zero_copy_only=False).astype(bool).tolist(); Bs = pc.equal(b.column('side'), 'bid').to_numpy(zero_copy_only=False).astype(bool).tolist(); Ps = b.column('price').to_numpy().tolist(); As = b.column('amount').to_numpy().tolist()
        for i in range(len(Ls)):
            L = Ls[i]; f = Fs[i]
            if max_rows is not None and st['rows'] >= max_rows: break
            st['rows'] += 1
            if started and (f != pf or ((not f) and L != pL)): end_msg(pL, pf)
            if not started:
                if not f: st['skipped_pre'] += 1; pL = L; pf = f; continue
                started = True
            if f and (pf is None or not pf or pf is False):
                for _, bk in V: bk.reset()
                st['resets'] += 1
            for _, bk in V: bk.upd(Bs[i], Ps[i], As[i])
            pL = L; pf = f
        else:
            if verbose and st['rows'] // 5_000_000 != done_rows // 5_000_000:
                el = time.time() - t0; print(f"   replayed {st['rows']:,} rows | {el:.0f}s | messages {st['messages']:,}")
            done_rows = st['rows']; continue
        break
    if started: end_msg(pL, pf)
    out = {'stats': st, 'cmp': cmp, 'timeline': TL, 'depth': depth, 'tick_est': {k: tick_of(bk) for k, bk in V}, 'fp': {k: bk.fingerprint() for k, bk in V}, 'final_levels': {k: (len(bk.bids), len(bk.asks)) for k, bk in V}, 'seconds': time.time() - t0}
    return out
QUICK_TEST_ROWS = None      # leave None for the full day. For a quick dry run on a small file set e.g. 2_000_000.
S25_STRIDE = max(1, (DL['book_snapshot_25']['lines_incl_header'] - 1) // 20000) if 'book_snapshot_25' in DL else 1
t0 = time.time(); SAMP, N_S25 = load_snapshot25_sample(S25_STRIDE); tick('load_snapshot25_sample', t0); print(f'book_snapshot_25 rows: {N_S25:,} | every {S25_STRIDE}-th row sampled -> {len(SAMP):,} distinct local_timestamps to compare')
t0 = time.time(); RP = run_replay(max_rows=QUICK_TEST_ROWS, run_unpruned=True, samp=SAMP, collect=True, run_pruned=False); tick('replay', t0)
st = RP['stats']; print({k: (dict(v) if isinstance(v, Counter) else v) for k, v in st.items()}); print('final levels (bids, asks):', RP['final_levels'], '| estimated tick size:', RP['tick_est'])

MSG = {k: np.frombuffer(v['lts'], dtype=np.int64) for k, v in RP['timeline'].items()}
n_msg = RP['stats']['messages']
CROSS = {}
for k in RP['timeline']:
    c = RP['stats']['crossed'][k]; l = RP['stats']['locked'][k]; CROSS[k] = {'messages_checked': n_msg, 'crossed_messages': c, 'locked_messages': l, 'crossed_share': c / max(n_msg, 1), 'locked_share': l / max(n_msg, 1), 'crossed_plus_locked_share': (c + l) / max(n_msg, 1), 'max_consecutive_crossed': RP['stats']['max_cross_run'][k]}
display(pd.DataFrame(CROSS).T); REPORT['crossed'] = CROSS
def sh(a, b): return (a / b) if b else None
CMP = {}
for k, c in RP['cmp'].items():
    matched = c['checkpoints'] + c['excluded_crossed_or_locked'] + c['empty_book_skipped']; sd = np.array(c['spread_diff'], float) if c['spread_diff'] else np.array([np.nan])
    d = {'snapshot25_rows_total': (N_S25 if SAMP else 0), 'snapshot25_rows_sampled': S25_META.get('rows_sampled'), 'distinct_local_timestamps_sampled': len(SAMP), 'matched_to_a_completed_message': matched, 'unmatched_no_message_with_equal_local_timestamp': len(SAMP) - matched,
         'excluded_book_crossed_or_locked': c['excluded_crossed_or_locked'], 'excluded_book_empty': c['empty_book_skipped'], 'ALIGNED_CHECKPOINTS': c['checkpoints']}
    for N in TOPNS:
        d[f'top{N}_price_agreement_(levels)'] = sh(c['p'][N], c['lv'][N]); d[f'top{N}_quantity_agreement_(price AND qty equal)'] = sh(c['q'][N], c['lv'][N]); d[f'top{N}_quantity_given_price_match'] = sh(c['q'][N], c['p'][N]); d[f'top{N}_checkpoints_all_prices_exact'] = sh(c['all_p'][N], c['checkpoints']); d[f'top{N}_checkpoints_all_prices_and_qty_exact'] = sh(c['all_pq'][N], c['checkpoints'])
    d.update({'best_bid_and_ask_both_equal': sh(c['best_ok'], c['checkpoints']), 'spread_exactly_equal': sh(c['spread_equal'], c['checkpoints']), 'spread_abs_diff_bps_median': float(np.nanmedian(sd)), 'spread_abs_diff_bps_p99': float(np.nanpercentile(sd, 99)),
              'worst_hours_(share_of_checkpoints_all25_exact)': sorted([(h, round(c['by_hour_ok'][h] / c['by_hour'][h], 3), c['by_hour'][h]) for h in c['by_hour']], key=lambda x: x[1])[:5]})
    CMP[k] = d
display(pd.DataFrame(CMP).T.T); REPORT['cmp_snapshot25'] = CMP
cov = {}
L_ = MSG['unpruned']; mins = np.unique((L_ - DAY0_US) // 60_000_000); mins = mins[(mins >= 0) & (mins < 1440)]; d_ = np.diff(L_) if len(L_) > 1 else np.array([0]); big = np.sort(d_)[-8:][::-1]
cov = {'messages': int(len(L_)), 'minutes_with_a_message_of_1440': int(len(mins)), 'coverage_of_day_by_minutes': len(mins) / 1440, 'first_message_utc': iso_us(L_.min()) if len(L_) else None, 'last_message_utc': iso_us(L_.max()) if len(L_) else None,
       'messages_per_second_mean': len(L_) / max((L_.max() - L_.min()) / 1e6, 1) if len(L_) else None, 'gaps_between_messages_over': {kk: int((d_ > th).sum()) for kk, th in GAP_TH.items()}, 'largest_message_gaps_s': [round(float(x) / 1e6, 1) for x in big]}
print(json.dumps(cov, indent=1, default=jsonable)); REPORT['coverage'] = cov
DEPTH = {}
for k, rows in RP['depth'].items():
    if not rows: continue
    df = pd.DataFrame(rows, columns=['L', 'mid', 'spread_bps', 'n_bid_levels', 'n_ask_levels', 'depth10bps', 'depth25bps', 'levels_far_5pct']); df['hour'] = ((df.L - DAY0_US) // 3_600_000_000).astype(int); DEPTH[k] = df
REPORT['depth_series_summary'] = {k: {'spread_bps_median': float(v.spread_bps.median()), 'levels_start': [int(v.n_bid_levels.iloc[0]), int(v.n_ask_levels.iloc[0])], 'levels_end': [int(v.n_bid_levels.iloc[-1]), int(v.n_ask_levels.iloc[-1])], 'levels_max': [int(v.n_bid_levels.max()), int(v.n_ask_levels.max())], 'far_levels_max_(>5pct_from_mid)': int(v.levels_far_5pct.max()), 'depth10bps_median_btc': float(v.depth10bps.median()), 'depth25bps_median_btc': float(v.depth25bps.median()), 'samples': len(v)} for k, v in DEPTH.items()}
print(json.dumps(REPORT['depth_series_summary'], indent=1, default=jsonable))

REQ = {'download_total_mb': round(sum(d['bytes'] for d in DL.values()) / 1e6, 1), 'uncompressed_total_gb': round(sum(d['uncompressed_bytes'] for d in DL.values()) / 1e9, 2), 'rows_incremental_book_L2': SC['rows'], 'rows_trades': TR['rows'], 'peak_memory_gb': round(peak_mem_gb(), 2), 'stage_seconds': STAGES,
       'replay_seconds_both_variants': round(RP['seconds'], 1), 'replay_rows_per_second': round(RP['stats']['rows'] / max(RP['seconds'], 1)),
       'estimate_note': 'ETH and extra days will take roughly the same time/space per day ONLY IF their row counts are similar; this is an extrapolation, not a measurement.'}
print(json.dumps(REQ, indent=1, default=jsonable)); REPORT['requirements'] = REQ
TESTS = {'E': [], 'D': []}
def tcase(group, name, fn):
    try: ok, msg = fn()
    except Exception as e: ok, msg = False, f'EXC {type(e).__name__}: {str(e)[:200]}'
    TESTS[group].append((name, bool(ok), msg)); print(('PASS ' if ok else 'FAIL ') + name + ' | ' + str(msg))
# ---------------- E: notebook execution tests
def e01():
    now = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file() and not str(p.relative_to(BASE)).startswith(STAGE.name)}
    bad = [k for k in PIN0 if now.get(k) != PIN0[k]]; new = [k for k in now if k not in PIN0]; return (not bad and not new), (f'{len(PIN0)} pre-existing files unchanged, no new files outside staging' if not bad and not new else f'CHANGED {bad[:3]} NEW {new[:3]}')
def e02(): names = sorted(p.name for p in RAW.iterdir()); ok = all(SYMBOL in n and DAY in n and n.endswith('.csv.gz') for n in names) and len(names) == len(DL); return ok, f'{names}'
def e03():
    a = run_replay(max_rows=200_000, run_unpruned=True, samp=None, collect=False, verbose=False, run_pruned=False); b = run_replay(max_rows=200_000, run_unpruned=True, samp=None, collect=False, verbose=False, run_pruned=False); return a['fp'] == b['fp'] and a['stats']['messages'] == b['stats']['messages'], f"two runs identical, fingerprints {a['fp']}"
def e04(): return RP['stats']['checks'] == RP['stats']['messages'] == n_msg, f'book inspected exactly once per completed message ({n_msg:,}); never mid-message'
def e05():
    bk = Book(0); bids = [(100.0 - i * 0.1, 1.0 + i) for i in range(25)]; asks = [(100.1 + i * 0.1, 2.0 + i) for i in range(25)]
    for p, q in bids: bk.upd(True, p, q)
    for p, q in asks: bk.upd(False, p, q)
    snap = (np.array([p for p, _ in asks]), np.array([q for _, q in asks]), np.array([p for p, _ in bids]), np.array([q for _, q in bids])); r = compare_top(bk, snap); ok1 = all(r[N]['p'] == r[N]['lv'] == r[N]['q'] for N in TOPNS) and r['best_ok'] and r['spread_equal']
    bk.upd(True, bids[7][0], 99.0); r2 = compare_top(bk, snap); ok2 = r2[5]['all_pq'] and (not r2[10]['all_pq']) and r2[10]['p'] == r2[10]['lv'] and r2[10]['q'] == r2[10]['lv'] - 1
    snap2 = (snap[0], snap[1], snap[2].copy(), snap[3]); snap2[2][3] += 0.05; bk2 = Book(0); [bk2.upd(True, p, q) for p, q in bids]; [bk2.upd(False, p, q) for p, q in asks]; r3 = compare_top(bk2, snap2); ok3 = (not r3[5]['all_p']) and r3[5]['p'] == r3[5]['lv'] - 1
    return ok1 and ok2 and ok3, f'identical book -> 100%; one changed quantity at level 8 -> top-5 exact, top-10 quantity 9/10; one shifted price -> top-5 price 9/10 (checks {ok1},{ok2},{ok3})'
def e06():
    c = RP['cmp']['unpruned']; matched = c['checkpoints'] + c['excluded_crossed_or_locked'] + c['empty_book_skipped']; return (not SAMP) or matched <= len(SAMP), f'matched {matched} <= distinct sampled keys {len(SAMP)}; unmatched rows are reported, never forced onto neighbouring messages'
for nm, fn in (('E01 pre-existing research files unchanged', e01), ('E02 only the intended files downloaded', e02), ('E03 replay deterministic (200k rows, 2 runs)', e03), ('E04 crossed-book check only at completed messages', e04), ('E05 snapshot-comparison function correct on constructed books', e05), ('E06 snapshot alignment accounting consistent', e06)): tcase('E', nm, fn)
# ---------------- D: dataset integrity tests
def d01(): bad = [k for k, d in DL.items() if not d['gzip_ok'] or (PROBE[k]['bytes'] and PROBE[k]['bytes'] != d['bytes'])]; return not bad, ('sizes equal server Content-Length and every gzip stream decodes fully' if not bad else bad)
def d02(): return all(v['matches_documented_header'] for v in SCHEMA.values()), 'headers equal the documented Tardis headers'
def d03(): bad = [(k, DL[k]['lines_incl_header'] - 1, n) for k, n in (('incremental_book_L2', SC['rows']), ('trades', TR['rows'])) if DL[k]['lines_incl_header'] - 1 != n]; return not bad, ('streamed row counts == independent newline counts' if not bad else bad)
def d04():
    df = pd.read_csv(DL['trades']['path'], nrows=300_000, usecols=['timestamp', 'local_timestamp', 'price', 'amount']); ok = df.price.min() > 0 and df.amount.min() > 0 and DAY0_US - 86_400_000_000 <= int(df.timestamp.min()) and int(df.timestamp.max()) <= DAY1_US + 86_400_000_000
    return ok, f'pandas recompute on the first 300k trades: price>0, amount>0, timestamps within +-1 day of {DAY}'
def d05():
    pre = pd.read_csv(DL['incremental_book_L2']['path'], nrows=500_000, usecols=['is_snapshot']).is_snapshot; first = int(np.argmax(pre.to_numpy())) if pre.any() else None; sb = SC['snapshot_batches']
    return (first is None and not sb) or bool(sb and (first is None or sb[0]['start_row'] == first)), f'first snapshot row: pandas {first} vs scan {sb[0]["start_row"] if sb else None}'
def d06(): return SC['symbols'] == {SYMBOL: SC['rows']} and set(SC['exchanges']) == {EXCHANGE}, f"book symbols {SC['symbols']} exchanges {SC['exchanges']}"
def d07():
    tot = SC['rows'] + TR['rows']; out = SC['local_outside_day_rows'] + TR['local_outside_day_rows']; return SC['timestamp_unit_check_ok'] and out / max(tot, 1) <= 0.001, f'timestamps are microseconds; local_timestamp outside {DAY} UTC: {out:,} of {tot:,} rows ({out/max(tot,1):.4%}, limit 0.1%)'
def d08(): bad = {'book_nan_cells': SC['nan_cells'], 'book_negative_amount': SC['amount_negative'], 'book_price<=0': SC['price_le_zero'], 'trade_amount_min': TR['amount_min'], 'trade_price_min': TR['price_range'][0]}; ok = SC['nan_cells'] == 0 and SC['amount_negative'] == 0 and SC['price_le_zero'] == 0 and TR['amount_min'] > 0 and TR['price_range'][0] > 0; return ok, str(bad)
def d09(): ok = set(SC['sides']) <= {'bid', 'ask'} and set(TR['sides']) <= {'buy', 'sell', 'unknown'}; return ok, f"book sides {SC['sides']} | trade sides {TR['sides']}"
def d10(): sb = SC['snapshot_batches']; ok = bool(sb) and all(b['bid_rows'] > 0 and b['ask_rows'] > 0 for b in sb); return ok, f'{len(sb)} snapshot batch(es); every batch has both bid and ask rows' if ok else f'snapshot batches missing or one-sided: {sb[:3]}'
for nm, fn in (('D01 file sizes + gzip integrity', d01), ('D02 schema equals documentation', d02), ('D03 row counts: streamed vs newline count', d03), ('D04 independent pandas recompute on 300k trades', d04), ('D05 first snapshot row vs pandas', d05), ('D06 symbol/exchange columns constant', d06), ('D07 microsecond units and day coverage', d07), ('D08 no NaN/negative/non-positive values', d08), ('D09 side vocabulary', d09), ('D10 snapshot batches contain both sides', d10)): tcase('D', nm, fn)
OBS = {'book_exchange_ts_inversions': SC['ts_inversions'], 'book_local_ts_inversions': SC['local_inversions'], 'trade_local_ts_inversions': TR['local_inversions'], 'book_zero_amount_rows(level_removals)': SC['amount_zero_rows'], 'trade_id_gaps(>1)': TR['id_gaps(>1)'], 'trade_id_dup_or_backwards': TR['id_dup_or_backwards'],
       'note': 'Observations, not pass/fail. The replay uses file order, as Tardis documents (rows are in capture order); inversions are listed so you can judge.'}
print('\nDataset observations (not pass/fail):', OBS); REPORT['observations'] = OBS
print(f"\nE (execution): {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])} | D (dataset integrity): {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])} | R (reconstruction): see next section")

R = {}
HAVE_S25 = 'book_snapshot_25' in DL
for name in CROSS:
    c = CROSS[name]; m = CMP.get(name, {}); p25 = m.get('top25_price_agreement_(levels)'); q25 = m.get('top25_quantity_agreement_(price AND qty equal)'); nck = m.get('ALIGNED_CHECKPOINTS', 0)
    R[name] = {
     'R01 crossed+locked share <= 0.5%': (c['crossed_plus_locked_share'] <= 0.005, c['crossed_plus_locked_share']),
     'R02 top-25 price agreement >= 99% (aligned checkpoints)': ((p25 or 0) >= 0.99 if HAVE_S25 else None, p25),
     'R03 top-25 quantity agreement >= 99% (aligned checkpoints)': ((q25 or 0) >= 0.99 if HAVE_S25 else None, q25),
     'R04 aligned snapshot checkpoints >= 500': (nck >= 500 if HAVE_S25 else None, nck),
     'R05 >= 99% of the 1,440 minutes contain a book message': (cov['coverage_of_day_by_minutes'] >= 0.99, cov['coverage_of_day_by_minutes'])}
def r_verdict(name):
    d = R[name]; fails = [k for k, (ok, v) in d.items() if ok is False]; untested = [k for k, (ok, v) in d.items() if ok is None]
    if untested: return 'UNTESTED for ' + '; '.join(untested) + ' (book_snapshot_25 not downloaded)' + (' | FAILED: ' + '; '.join(fails) if fails else '')
    return ("CONSISTENT with the provider's own book_snapshot_25 (same provider/feed; NOT independent validation)" if not fails else 'NOT CONSISTENT -> failed: ' + '; '.join(fails))
RV = {k: r_verdict(k) for k in R}
REPORT['reconstruction_criteria'] = {k: {c: {'pass': ok, 'value': v} for c, (ok, v) in d.items()} for k, d in R.items()}; REPORT['reconstruction_verdict'] = RV
for k in R:
    print(k); [print('  ' + ('PASS ' if ok else ('FAIL ' if ok is False else 'N/A  ')) + c + ' -> ' + str(v)) for c, (ok, v) in R[k].items()]; print('  VERDICT:', RV[k])

ghost = {k: {'levels_end': REPORT['depth_series_summary'].get(k, {}).get('levels_end'), 'levels_max': REPORT['depth_series_summary'].get(k, {}).get('levels_max'), 'far_levels_max': REPORT['depth_series_summary'].get(k, {}).get('far_levels_max_(>5pct_from_mid)')} for k in DEPTH}
unc = ['One symbol and one day only: this says nothing about other days, ETHUSDT, or whether the free-day files are representative.',
       'The CSV has NO update-ID columns, so exchange sequence continuity (U/u) cannot be re-audited here; Tardis states it validates U/u and restarts its connection on a gap (documentation claim, not verified by this notebook).',
       "book_snapshot_25 is produced by Tardis from the same feed, so agreement with it shows consistency with Tardis' own reconstruction, not independence from the exchange or scientific validation.",
       'The trade and depth streams are separate WebSocket feeds; the trade-vs-book figures are a diagnostic of plausibility and timing, not proof of exact ordering, and they gate nothing.',
       'The terms do not say whether publishing thesis results from this data is allowed; ask Tardis in writing. Raw files and trained models must not be given to third parties.']
L = []; P = L.append
def fm(x, f='{:.4%}'): return f.format(x) if isinstance(x, (int, float)) and x == x else str(x)
P('=' * 100); P(f'FINAL DATASET AUDIT SHARE BLOCK - Tardis.dev Binance Spot {SYMBOL} {DAY} (v1; reference book = full-depth)'); P('=' * 100)
P(f'run: {RUN_STARTED} | folder: {OUT_FINAL} | free-access download, no API key: {API_KEY is None}')
P('Three kinds of result are kept apart below: [E] notebook execution tests, [D] dataset integrity tests, [R] order-book reconstruction assessment.')
P('PASSING E AND D DOES NOT MEAN THE RECONSTRUCTED BOOK IS VALIDATED. Only section [R] speaks about the book, and only as consistency with the provider\'s own snapshots.')
P('\n[A] FILES'); [P(f"  {d['file']}: {d['bytes']/1e6:,.1f} MB compressed | {d['uncompressed_bytes']/1e9:.2f} GB uncompressed | lines {d['lines_incl_header']-1:,} | gzip_ok {d['gzip_ok']} | sha256 {d['sha256'][:16]}...") for d in DL.values()]
P('  schemas match documentation: ' + str({k: v['matches_documented_header'] for k, v in SCHEMA.items()}))
P('\n[B] TIMESTAMPS (microseconds)'); P(f"  book exchange ts: {SC['timestamp_min']} -> {SC['timestamp_max']} | local ts: {SC['local_min']} -> {SC['local_max']} | inversions ts/local: {SC['ts_inversions']}/{SC['local_inversions']}")
P(f"  trades local ts: {TR['local_min']} -> {TR['local_max']} | trade local-ts inversions {TR['local_inversions']} | trade id gaps {TR['id_gaps(>1)']} dup/back {TR['id_dup_or_backwards']} | latency (local-exchange) trades median {TR['local_minus_exchange_us_median']:.0f} us p99 {TR['local_minus_exchange_us_p99']:.0f} us | book median {SC['book_local_minus_exchange_us_median']:.0f} us p99 {SC['book_local_minus_exchange_us_p99']:.0f} us")
P(f"  book rows {SC['rows']:,} | trades {TR['rows']:,} | side values {SC['sides']} | zero-amount rows {SC['amount_zero_rows']:,} | negative {SC['amount_negative']} | NaN cells {SC['nan_cells']} | price<=0 {SC['price_le_zero']}")
P('\n[C] COVERAGE'); P('  ' + json.dumps(cov, default=jsonable)); P(f"  book message gaps (local ts, row-level) >1s/10s/60s/300s: {SC['gap_counts']} | largest: {SC['largest_gaps_s'][:5]}")
P('\n[D-snap] SNAPSHOT BOUNDARIES'); P(f"  rows before first snapshot (to be skipped): {SC['rows_before_first_snapshot']:,} | snapshot rows total {SC['snapshot_rows']:,} | snapshot batches: {len(SC['snapshot_batches'])}")
for b in SC['snapshot_batches'][:12]: P(f"    start_row {b['start_row']:,} at {iso_us(b['local_ts'])}: {b['rows']:,} rows (bids {b['bid_rows']:,} / asks {b['ask_rows']:,})")
P('\n' + '-' * 100); P('[E] NOTEBOOK EXECUTION TESTS (did the code behave?)'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['E']]
P(f"  => E: {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])} passed")
P('\n' + '-' * 100); P('[D] DATASET INTEGRITY TESTS (are the files intact and well-formed?)'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['D']]
P(f"  => D: {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])} passed"); P('  observations (not pass/fail): ' + json.dumps(OBS, default=jsonable))
P('\n' + '-' * 100); P('[R] ORDER-BOOK RECONSTRUCTION ASSESSMENT (independent of E and D; same-provider consistency only)')
for k in CROSS: P(f"  {k}: messages {CROSS[k]['messages_checked']:,} | crossed {CROSS[k]['crossed_messages']:,} ({CROSS[k]['crossed_share']:.4%}) | locked {CROSS[k]['locked_messages']:,} ({CROSS[k]['locked_share']:.4%}) | crossed+locked {CROSS[k]['crossed_plus_locked_share']:.4%} | longest crossed run {CROSS[k]['max_consecutive_crossed']} | final levels {RP['final_levels'][k]} | resets {RP['stats']['resets']}")
P('  snapshot comparison rule: book_snapshot_25 row vs rebuilt book after the completed message with EXACTLY equal local_timestamp; unmatched rows counted, crossed/locked/empty moments excluded and counted.')
for k, m in CMP.items():
    P(f"  vs book_snapshot_25 [{k}]: rows {m['snapshot25_rows_total']:,} (sampled {m['snapshot25_rows_sampled']}) | matched to a completed message {m['matched_to_a_completed_message']:,} | unmatched {m['unmatched_no_message_with_equal_local_timestamp']:,} | excluded crossed/locked {m['excluded_book_crossed_or_locked']:,} | excluded empty {m['excluded_book_empty']:,} | ALIGNED CHECKPOINTS {m['ALIGNED_CHECKPOINTS']:,}")
    for N in TOPNS: P(f"     top-{N}: price agreement {fm(m[f'top{N}_price_agreement_(levels)'])} | quantity agreement (price+qty) {fm(m[f'top{N}_quantity_agreement_(price AND qty equal)'])} | qty given price match {fm(m[f'top{N}_quantity_given_price_match'])} | checkpoints all-prices-exact {fm(m[f'top{N}_checkpoints_all_prices_exact'])} / all-prices+qty-exact {fm(m[f'top{N}_checkpoints_all_prices_and_qty_exact'])}")
    P(f"     best bid&ask equal {fm(m['best_bid_and_ask_both_equal'])} | spread exactly equal {fm(m['spread_exactly_equal'])} | |spread diff| median {fm(m['spread_abs_diff_bps_median'], '{:.4f}')} bps p99 {fm(m['spread_abs_diff_bps_p99'], '{:.4f}')} bps | worst hours {m['worst_hours_(share_of_checkpoints_all25_exact)']}")
P('  stale-level indicators (levels per side end / max / max beyond 5% of mid): ' + json.dumps(ghost, default=jsonable)); P('  depth/spread summary: ' + json.dumps(REPORT['depth_series_summary'], default=jsonable))
P('  frozen reconstruction criteria:')
for k, d in R.items():
    P(f'   variant {k}:'); [P(f"     {'PASS' if ok else ('FAIL' if ok is False else 'N/A ')}  {c}  -> {v}") for c, (ok, v) in d.items()]; P('     RECONSTRUCTION VERDICT: ' + RV[k])
P('\n[G] PROCESSING REQUIREMENTS'); P('  ' + json.dumps(REQ, default=jsonable))
P('\n[I] LICENCE REMINDER: internal research/educational use; no redistribution of raw data; no giving trained models to third parties; publication of results not addressed in the terms -> ask Tardis in writing.')
P('\nUNCERTAINTIES'); [P('  - ' + u) for u in unc]
GATE_OK_E = all(t[1] for t in TESTS['E']); GATE_OK_D = all(t[1] for t in TESTS['D'])
P(f"\nFOLDER GATE (E and D only): E {'ok' if GATE_OK_E else 'FAILED'} | D {'ok' if GATE_OK_D else 'FAILED'} -> the R verdict does not decide whether files are saved.")
P('Nothing else was downloaded by this job. No synthetic events, graphs or models were created. Existing research artifacts were not modified. WAITING FOR YOUR APPROVAL before any next step.'); P('=' * 100)
SHARE = '\n'.join(L)
STAGE.mkdir(exist_ok=True); (STAGE / 'reports').mkdir(exist_ok=True)
REPORT['tests'] = TESTS; REPORT['unc'] = unc; REPORT['finished'] = UTC()
dump_json(REPORT, STAGE / 'reports' / 'audit_report.json'); (STAGE / 'reports' / 'share_block.txt').write_text(SHARE, encoding='utf-8')
pd.DataFrame(SC['snapshot_batches']).to_csv(STAGE / 'reports' / 'snapshot_batches.csv', index=False)
for k, df in DEPTH.items(): df.to_parquet(STAGE / 'reports' / f'depth_series_{k}.parquet', index=False)
MAN = {str(p.relative_to(STAGE)): sha256_file(p) for p in sorted(STAGE.rglob('*')) if p.is_file()}; dump_json(MAN, STAGE / 'manifest.json')
okm = all(sha256_file(STAGE / k) == v for k, v in MAN.items()); all_ok = GATE_OK_E and GATE_OK_D and okm
if all_ok: os.rename(STAGE, OUT_FINAL); print('\nSAVED ->', OUT_FINAL, '(files intact; this is NOT a statement that the book is validated)')
else: print('\n!! E or D tests FAILED (or manifest mismatch) -> staging kept at', STAGE, '(not renamed). Share this block anyway.')
print(SHARE); print(f'manifest verified: {okm} ({len(MAN)} files)')

Writing tardis_body_audit.py


In [ ]:
%%writefile tardis_body_walk.py
import os, sys, io, re, gc, json, time, math, gzip, uuid, shutil, hashlib, heapq, warnings, resource
from array import array
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 100)
pass      # the driver cell has already mounted Drive
# NOTE: this notebook deliberately imports no HTTP library and never touches the network.
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
assert BASE.is_dir(), f'Drive folder not found: {BASE}'
SRC = BASE / f'TGNN_Tardis_Binance_Spot_{JOB_SYMBOL}_{JOB_DAY}_v1'
OUT_FINAL = BASE / f'TGNN_Tardis_WalkBook_{JOB_SYMBOL}_{JOB_DAY}_v1'
DIAG08 = BASE / f'TGNN_Tardis_TradeDiag_{JOB_SYMBOL}_{JOB_DAY}_v1' / 'reports' / 'trade_diagnostic_report.json'     # optional: used only for a cross-notebook consistency test
STAGE = BASE / ('.tardis_walk_stage_' + uuid.uuid4().hex[:8])
assert SRC.is_dir(), f'Source folder from notebook 07 not found: {SRC}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
EXCHANGE = 'binance'; SYMBOL = JOB_SYMBOL; DAY = JOB_DAY
DAY0_US = int(pd.Timestamp(DAY, tz='UTC').timestamp() * 1_000_000); DAY1_US = DAY0_US + 86_400_000_000
FILES = {dt: SRC / 'raw' / f'{EXCHANGE}_{dt}_{DAY}_{SYMBOL}.csv.gz' for dt in ('incremental_book_L2', 'trades', 'book_snapshot_25')}
for k in ('incremental_book_L2', 'trades'): assert FILES[k].is_file(), f'missing {FILES[k]}'
HAVE_S25 = FILES['book_snapshot_25'].is_file()
UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); STAGES = {}; REPORT = {}
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def peak_mem_gb(): return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
def tick_(name, t0): STAGES[name] = {'seconds': round(time.time() - t0, 1), 'peak_mem_gb_so_far': round(peak_mem_gb(), 2)}; print(f'[{name}] {STAGES[name]["seconds"]} s | peak memory so far {STAGES[name]["peak_mem_gb_so_far"]} GB')
def sh(a, b): return (a / b) if b else float('nan')
t0 = time.time(); PIN0 = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file()}
print(f'fingerprinted {len(PIN0)} existing files (including the 07 output folder); they will be re-checked at the end'); tick_('fingerprint', t0)
MAN = json.loads((SRC / 'manifest.json').read_text()); SRC_REPORT = json.loads((SRC / 'reports' / 'audit_report.json').read_text()); SRC_SHARE = (SRC / 'reports' / 'share_block.txt').read_text()
print('source folder:', SRC); print('book_snapshot_25 present:', HAVE_S25)

t0 = time.time()
SHA_CHECK = {}
for dt, p in FILES.items():
    if not p.is_file(): continue
    key = 'raw/' + p.name; SHA_CHECK[dt] = {'file': p.name, 'recorded': MAN.get(key), 'now': sha256_file(p)}; SHA_CHECK[dt]['equal'] = SHA_CHECK[dt]['recorded'] == SHA_CHECK[dt]['now']
    print(dt, 'sha256 equal to manifest:', SHA_CHECK[dt]['equal'])
tick_('sha256_recheck', t0)
t0 = time.time()
try: tr_tbl = pacsv.read_csv(FILES['trades'], convert_options=pacsv.ConvertOptions(column_types={'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64(), 'id': pa.int64(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()}))
except Exception as e: raise RuntimeError('trade ids are not plain integers -> STOP and paste this to Claude: ' + str(e)[:200])
tE = tr_tbl.column('timestamp').to_numpy(); tL = tr_tbl.column('local_timestamp').to_numpy(); tid = tr_tbl.column('id').to_numpy(); tpx = tr_tbl.column('price').to_numpy(); tam = tr_tbl.column('amount').to_numpy()
side_s = tr_tbl.column('side'); tside = np.full(len(tE), 2, np.int8); tside[pc.equal(side_s, 'buy').to_numpy(zero_copy_only=False).astype(bool)] = 0; tside[pc.equal(side_s, 'sell').to_numpy(zero_copy_only=False).astype(bool)] = 1
NT = len(tE); del tr_tbl; gc.collect(); tick_('load_trades', t0); print(f'trades loaded: {NT:,} | buy {int((tside==0).sum()):,} sell {int((tside==1).sum()):,} other {int((tside==2).sum()):,}')
TRADES_SORTED_LOCAL = bool(np.all(np.diff(tL) >= 0)); TRADES_SORTED_EXCH = bool(np.all(np.diff(tE) >= 0)); print('trades sorted by local ts:', TRADES_SORTED_LOCAL, '| by exchange ts:', TRADES_SORTED_EXCH)
if not TRADES_SORTED_LOCAL: raise RuntimeError('trades are not in local-timestamp order: STOP and paste this to Claude.')
BOOK_TYPES = {'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64(), 'is_snapshot': pa.bool_(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()}
def book_reader(block_mb=48): return pacsv.open_csv(FILES['incremental_book_L2'], read_options=pacsv.ReadOptions(block_size=block_mb << 20), convert_options=pacsv.ConvertOptions(column_types=BOOK_TYPES, strings_can_be_null=False, true_values=['true', 'True', 'TRUE'], false_values=['false', 'False', 'FALSE']))
# tick size from the data itself: the LARGEST "nice" step (1, 2, 5 x 10^k) on which every price in the first book block and in the trades lies
first_blk = next(iter(book_reader(16))); pr0 = np.unique(first_blk.column('price').to_numpy())
def off_grid_share(p, tk): x = p / tk; return float(np.mean(np.abs(x - np.rint(x)) > 1e-4))
CANDS = sorted({m * 10.0 ** k for k in range(-8, 3) for m in (1, 2, 5)}, reverse=True); probe_p = np.concatenate([pr0, tpx[:200_000]])
TICK = next((c for c in CANDS if off_grid_share(probe_p, c) == 0.0), None)
if TICK is None: raise RuntimeError('no tick size fits all prices: STOP and paste this to Claude.')
TICK = float(TICK); print('tick size inferred from the data:', TICK)
GRID = {'book_first_block_off_grid_share': off_grid_share(pr0, TICK), 'trades_off_grid_share': off_grid_share(tpx, TICK)}; print(GRID)
tpt = np.rint(tpx / TICK).astype(np.int64)

WALK_CAP = 30_000; TOL_ABS = 1e-7; TOL_REL = 1e-9
def walk_classify(dct, touch, dirn, opp, prices, amts, cap=WALK_CAP):
    # dct: {tick: displayed qty} for the side being hit; dirn = -1 for sell-takers (deeper = lower price), +1 for buy-takers
    n = len(prices); cls = [0] * n; Db = [0.0] * n; Di = [0.0] * n; Cp = [0.0] * n; dist = [0] * n; need = set()
    for i in range(n):
        d = (prices[i] - touch) * dirn; dist[i] = d
        if d < 0: cls[i] = 6 if ((dirn < 0 and prices[i] >= opp) or (dirn > 0 and prices[i] <= opp)) else 5
        elif d > cap: cls[i] = 7
        else: need.add(d)
    res = {}; k = 0; cum = 0.0
    for dd_ in sorted(need):
        while k < dd_: cum += dct.get(touch + dirn * k, 0.0); k += 1
        pk = touch + dirn * dd_; res[dd_] = (cum, cum + dct.get(pk, 0.0), pk in dct)
    C = 0.0
    for i in range(n):
        if cls[i] in (5, 6, 7): continue
        db, di, shown = res[dist[i]]; Db[i] = db; Di[i] = di; Cp[i] = C
        if not shown: cls[i] = 4; continue
        a = amts[i]; reach_ok = C >= db - TOL_ABS - TOL_REL * db; exh_ok = (C + a) <= di + TOL_ABS + TOL_REL * di
        cls[i] = 0 if (reach_ok and exh_ok) else 1 if (not reach_ok and exh_ok) else 2 if (reach_ok and not exh_ok) else 3
        C += a
    return cls, Db, Di, Cp, dist
CLS_NAMES = {0: 'consistent', 1: 'reach_violated(deeper_than_volume_explains)', 2: 'exhaustion_violated(more_than_displayed)', 3: 'both_violated', 4: 'price_not_displayed_level', 5: 'better_than_touch_inside_spread', 6: 'at_or_through_opposite_touch', 7: 'beyond_walk_cap', 8: 'no_valid_state', 9: 'unknown_side'}

class IBook:
    __slots__ = ('bids', 'asks', 'bh', 'ah')
    def __init__(s): s.bids = {}; s.asks = {}; s.bh = []; s.ah = []
    def reset(s): s.bids.clear(); s.asks.clear(); s.bh.clear(); s.ah.clear()
    def upd(s, is_bid, p, a):
        if is_bid:
            d = s.bids
            if a > 0:
                if p not in d: heapq.heappush(s.bh, -p)
                d[p] = a
            else: d.pop(p, None)
        else:
            d = s.asks
            if a > 0:
                if p not in d: heapq.heappush(s.ah, p)
                d[p] = a
            else: d.pop(p, None)
    def best(s):
        bh = s.bh; bd = s.bids
        while bh and -bh[0] not in bd: heapq.heappop(bh)
        ah = s.ah; ad = s.asks
        while ah and ah[0] not in ad: heapq.heappop(ah)
        if len(bh) > 4 * len(bd) + 4000: s.bh = bh = [-p for p in bd]; heapq.heapify(bh)
        if len(ah) > 4 * len(ad) + 4000: s.ah = ah = list(ad); heapq.heapify(ah)
        return (-bh[0] if bh else 0), (ah[0] if ah else 0)
TL_ = tL.tolist(); TP_ = tpt.tolist(); TS_ = tside.tolist(); TA_ = tam.tolist()

def load_s25_best(stride):
    out = {}; n = 0
    if not HAVE_S25: return out, 0
    rd = pacsv.open_csv(FILES['book_snapshot_25'], read_options=pacsv.ReadOptions(block_size=48 << 20), convert_options=pacsv.ConvertOptions(include_columns=['local_timestamp', 'asks[0].price', 'bids[0].price'], column_types={'local_timestamp': pa.int64(), 'asks[0].price': pa.float64(), 'bids[0].price': pa.float64()}))
    for b in rd:
        L = b.column('local_timestamp').to_numpy(); A = b.column('asks[0].price').to_numpy(); B = b.column('bids[0].price').to_numpy(); idx = np.arange((-n) % stride, len(L), stride); n += len(L)
        At = np.rint(A[idx] / TICK).astype(np.int64).tolist(); Bt = np.rint(B[idx] / TICK).astype(np.int64).tolist()
        for l, a, bb in zip(L[idx].tolist(), At, Bt): out[l] = (a, bb)
    return out, n

def run_pass(max_rows=None, samp=None, verbose=True):
    rd = book_reader(); bk = IBook(); n = NT
    cls = np.full(n, 8, np.int8); dist = np.zeros(n, np.int32); Dbq = np.zeros(n); Diq = np.zeros(n); Cpq = np.zeros(n); gsz = np.zeros(n, np.int32); gvol = np.zeros(n); post_ok = np.full(n, -1, np.int8); age = np.full(n, -1, np.int64); touchq = np.zeros(n)
    ML = array('q'); ME = array('q'); MB = array('q'); MA = array('q')
    st = {'rows': 0, 'skipped_pre': 0, 'messages': 0, 'resets': 0, 'crossed': 0, 'locked': 0, 'snap_checked': 0, 'snap_best_equal': 0, 'trades_after_last_message': 0, 'groups': 0}
    S = {'j': 0, 'L': None, 'bb': 0, 'ba': 0}; pend = []
    def flush(Lcut):
        j = S['j']
        if j >= n or TL_[j] > Lcut: return
        j1 = j
        while j1 < n and TL_[j1] <= Lcut: j1 += 1
        bb = S['bb']; ba = S['ba']; cL = S['L']; valid = cL is not None and bb > 0 and ba > 0 and bb < ba
        if not valid:
            for t in range(j, j1): cls[t] = 9 if TS_[t] == 2 else 8
            S['j'] = j1; return
        sells = []; buys = []
        for t in range(j, j1):
            s = TS_[t]; age[t] = TL_[t] - cL
            if s == 1: sells.append(t)
            elif s == 0: buys.append(t)
            else: cls[t] = 9
        for lst, dct, touch, dirn, opp in ((sells, bk.bids, bb, -1, ba), (buys, bk.asks, ba, 1, bb)):
            if not lst: continue
            pr = [TP_[t] for t in lst]; am = [TA_[t] for t in lst]; c, Db, Di, Cp, ds = walk_classify(dct, touch, dirn, opp, pr, am); st['groups'] += 1; vol = sum(am); tq0 = dct.get(touch, 0.0); pw = None
            for q, t in enumerate(lst):
                cls[t] = c[q]; dist[t] = ds[q]; Dbq[t] = Db[q]; Diq[t] = Di[q]; Cpq[t] = Cp[q]; gsz[t] = len(lst); gvol[t] = vol; touchq[t] = tq0
                if ds[q] >= 0 and c[q] != 7 and (pw is None or ds[q] > pw): pw = ds[q]
            if pw is not None: pend.append((lst, dirn, touch + dirn * pw))
        S['j'] = j1
    def end_msg(L, E):
        st['messages'] += 1; bb, ba = bk.best(); ML.append(L); ME.append(E); MB.append(bb); MA.append(ba)
        if bb and ba:
            if bb > ba: st['crossed'] += 1
            elif bb == ba: st['locked'] += 1
        S['L'] = L; S['bb'] = bb; S['ba'] = ba
        if pend:
            for lst, dirn, pw in pend:
                ok = (bb > 0 and bb <= pw) if dirn < 0 else (ba > 0 and ba >= pw)
                post_ok[lst] = 1 if ok else 0
            pend.clear()
        if samp is not None and L in samp and bb and ba and bb < ba:
            a0, b0 = samp[L]; st['snap_checked'] += 1; st['snap_best_equal'] += int(a0 == ba and b0 == bb)
    started = False; pL = None; pf = None; pE = None; stop = False; t0 = time.time(); done = 0
    for b in rd:
        Ls = b.column('local_timestamp').to_numpy().tolist(); Es = b.column('timestamp').to_numpy().tolist(); Fs = b.column('is_snapshot').to_numpy(zero_copy_only=False).astype(bool).tolist()
        Bs = pc.equal(b.column('side'), 'bid').to_numpy(zero_copy_only=False).astype(bool).tolist(); Ps = np.rint(b.column('price').to_numpy() / TICK).astype(np.int64).tolist(); As = b.column('amount').to_numpy().tolist()
        for i in range(len(Ls)):
            if max_rows is not None and st['rows'] >= max_rows: stop = True; break
            L = Ls[i]; f = Fs[i]; st['rows'] += 1
            if started and (f != pf or ((not f) and L != pL)):
                end_msg(pL, pE); flush(L)
            if not started:
                if not f: st['skipped_pre'] += 1; pL = L; pf = f; pE = Es[i]; continue
                started = True; flush(L)
            if f and (pf is None or not pf): bk.reset(); st['resets'] += 1
            bk.upd(Bs[i], Ps[i], As[i]); pL = L; pf = f; pE = Es[i]
        if stop: break
        if verbose and st['rows'] // 5_000_000 != done // 5_000_000: print(f"   replayed {st['rows']:,} rows | {time.time()-t0:.0f}s | messages {st['messages']:,} | trades evaluated {S['j']:,}")
        done = st['rows']
    if started and not stop:
        end_msg(pL, pE); st['trades_after_last_message'] = n - S['j']; flush(2 ** 62)
    asarr = lambda a: np.frombuffer(a, np.int64).copy() if len(a) else np.zeros(0, np.int64)
    return {'stats': st, 'cls': cls, 'dist': dist, 'Db': Dbq, 'Di': Diq, 'Cp': Cpq, 'gsz': gsz, 'gvol': gvol, 'post_ok': post_ok, 'age': age, 'touchq': touchq, 'ML': asarr(ML), 'ME': asarr(ME), 'MB': asarr(MB), 'MA': asarr(MA), 'trades_evaluated': S['j'], 'final_levels': (len(bk.bids), len(bk.asks)), 'seconds': time.time() - t0}

S25_STRIDE = 40; t0 = time.time(); SAMP, N_S25 = load_s25_best(S25_STRIDE); tick_('load_snapshot25_best_sample', t0); print(f'book_snapshot_25 rows: {N_S25:,} | every {S25_STRIDE}-th row sampled -> {len(SAMP):,} distinct local_timestamps')
t0 = time.time(); PS = run_pass(samp=SAMP); tick_('replay_walk_the_book', t0)
st = PS['stats']; print(st); print('final levels (bids, asks):', PS['final_levels'], '| trades evaluated:', PS['trades_evaluated'], 'of', NT)
cls = PS['cls']; MLn, MEn, MBn, MAn = PS['ML'], PS['ME'], PS['MB'], PS['MA']

valid = (cls <= 7); NV = int(valid.sum()); dist = PS['dist']; gsz = PS['gsz']; Q = valid & (dist >= 2); NQ = int(Q.sum())
def shares(mask):
    c = np.bincount(cls[mask], minlength=10)[:8]; return c, c / max(int(c.sum()), 1)
rows = {}
for nm, m in (('all', valid), ('sell_taker', valid & (tside == 1)), ('buy_taker', valid & (tside == 0)), ('deep_Q_(>=2_ticks_deeper)', Q), ('not_deep', valid & ~Q)):
    c, s_ = shares(m); rows[nm + '_count'] = c; rows[nm + '_share'] = s_
CLSTAB = pd.DataFrame(rows, index=[CLS_NAMES[i] for i in range(8)]); display(CLSTAB.style.format({c: '{:.3%}' for c in CLSTAB.columns if c.endswith('share')}))
W = float(CLSTAB['all_share'].iloc[0]); I_ = float(CLSTAB['all_share'].iloc[5] + CLSTAB['all_share'].iloc[6]); Qc = float(CLSTAB['deep_Q_(>=2_ticks_deeper)_share'].iloc[0])
Q_NOT_DISPLAYED = float(CLSTAB['deep_Q_(>=2_ticks_deeper)_share'].iloc[4]); print(f'usable trades {NV:,} | W (consistent, all) = {W:.2%} | I (better than touch / through opposite) = {I_:.2%} | deep trades Q = {NQ:,} ({NQ/max(NV,1):.2%}) | Qc (deep trades that are consistent) = {Qc:.2%}')
bins = [(1, 1, '1 trade'), (2, 3, '2-3 trades'), (4, 10, '4-10 trades'), (11, 10 ** 9, '>10 trades')]; BR = []
for lo_, hi_, lab in bins:
    m = valid & (gsz >= lo_) & (gsz <= hi_); n_ = int(m.sum()); r = {'burst_size_(same_side_trades_in_interval)': lab, 'trades': n_, 'share_of_usable': sh(n_, NV), 'P(deep)': sh(int((Q & m).sum()), n_)}
    for k_, nm in ((0, 'consistent'), (1, 'reach_violated'), (2, 'exhaustion_violated'), (3, 'both'), (4, 'not_displayed')): r[nm] = sh(int(((cls == k_) & m).sum()), n_)
    r['better_than_touch_or_through'] = sh(int((((cls == 5) | (cls == 6)) & m).sum()), n_); BR.append(r)
BURST = pd.DataFrame(BR); display(BURST.style.format({c: '{:.2%}' for c in BURST.columns if c not in ('burst_size_(same_side_trades_in_interval)', 'trades')}))

amt = tam; Db, Di, Cp = PS['Db'], PS['Di'], PS['Cp']; pok = PS['post_ok']
def qs(a): return [float(x) for x in np.percentile(a, [50, 75, 90, 99])] if len(a) else None
m1 = (cls == 1) | (cls == 3); m2 = (cls == 2) | (cls == 3)
short = (Db - Cp)[m1]; excess = (Cp + amt - Di)[m2]
MAG = {'reach_violated_trades': int(m1.sum()), 'shortfall_BTC_quantiles(50,75,90,99)': qs(short), 'shortfall_as_share_of_better_depth_quantiles(50,75,90,99)': qs((short / np.maximum(Db[m1], 1e-12))) if m1.any() else None,
       'exhaustion_violated_trades': int(m2.sum()), 'excess_BTC_quantiles(50,75,90,99)': qs(excess), 'median_trade_amount_BTC': float(np.median(tam)),
       'interval_volume_over_touch_qty_quantiles(50,75,90,99)_for_deep_trades': qs((PS['gvol'] / np.maximum(PS['touchq'], 1e-12))[Q & (PS['touchq'] > 0)])}
POST = {}
for k_, nm in ((0, 'consistent'), (1, 'reach_violated'), (2, 'exhaustion_violated'), (3, 'both'), (4, 'not_displayed')):
    m = (cls == k_) & (pok >= 0); POST[nm] = {'trades_with_post_state': int(m.sum()), 'touch_moved_to_or_through_deepest_print': sh(int((pok[m] == 1).sum()), int(m.sum()))}
m = Q & (pok >= 0); POST['all_deep_Q'] = {'trades_with_post_state': int(m.sum()), 'touch_moved_to_or_through_deepest_print': sh(int((pok[m] == 1).sum()), int(m.sum()))}
vol_w = {CLS_NAMES[k_]: float(tam[(cls == k_)].sum() / max(tam[valid].sum(), 1e-12)) for k_ in range(8)}
print(json.dumps({'magnitudes': MAG, 'post_interval_touch': POST, 'volume_weighted_class_shares': vol_w}, indent=1, default=jsonable))
# time of day and volatility (consistent share)
hour = np.clip((tL - DAY0_US) // 3_600_000_000, 0, 23).astype(int); HOUR = pd.DataFrame({'h': hour[valid], 'c': (cls[valid] == 0)}).groupby('h').c.agg(['mean', 'size']).rename(columns={'mean': 'consistent_share', 'size': 'trades'})
mn = np.clip((tL - DAY0_US) // 60_000_000, 0, 1439).astype(int); g = pd.DataFrame({'m': mn, 'p': tpx}).groupby('m').p; rng_bps = ((g.max() - g.min()) / g.mean() * 1e4).rename('rng_bps')
dfv = pd.DataFrame({'m': mn[valid], 'c': (cls[valid] == 0)}).join(rng_bps, on='m'); dfv['vol_quintile'] = pd.qcut(dfv.rng_bps.rank(method='first'), 5, labels=['Q1 calmest', 'Q2', 'Q3', 'Q4', 'Q5 most volatile']); VOL = dfv.groupby('vol_quintile', observed=True).agg(consistent_share=('c', 'mean'), trades=('c', 'size'))
display(HOUR.style.format({'consistent_share': '{:.2%}'})); display(VOL.style.format({'consistent_share': '{:.2%}'}))
REPORT['classes'] = {'table': CLSTAB.to_dict(), 'W_consistent_all': W, 'I_better_or_through': I_, 'Qc_deep_consistent': Qc, 'usable_trades': NV, 'deep_trades': NQ}; REPORT['burst_table'] = BURST.to_dict('records'); REPORT['magnitudes'] = MAG; REPORT['post_interval_touch'] = POST; REPORT['volume_weighted'] = vol_w; REPORT['time_of_day'] = HOUR.reset_index().to_dict('records'); REPORT['volatility'] = VOL.reset_index().to_dict('records')

import random
TESTS = {'E': [], 'D': []}
def tcase(group, name, fn):
    try: ok, msg = fn()
    except Exception as e: ok, msg = False, f'EXC {type(e).__name__}: {str(e)[:200]}'
    TESTS[group].append((name, bool(ok), msg)); print(('PASS ' if ok else 'FAIL ') + name + ' | ' + str(msg))
def brute(dct, touch, dirn, opp, prices, amts, cap=WALK_CAP):
    # independent re-implementation: no incremental walk; depth better than p is summed directly from the dictionary
    out = []; C = 0.0
    for p, a in zip(prices, amts):
        d = (p - touch) * dirn
        if d < 0: out.append(6 if ((dirn < 0 and p >= opp) or (dirn > 0 and p <= opp)) else 5); continue
        if d > cap: out.append(7); continue
        if p not in dct: out.append(4); continue
        db = sum(q for pk, q in dct.items() if (pk - p) * dirn < 0); di = db + dct[p]
        reach = C >= db - TOL_ABS - TOL_REL * db; exh = (C + a) <= di + TOL_ABS + TOL_REL * di
        out.append(0 if (reach and exh) else 1 if (not reach and exh) else 2 if (reach and not exh) else 3); C += a
    return out
def e01():
    now = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file() and not str(p.relative_to(BASE)).startswith(STAGE.name)}
    bad = [k for k in PIN0 if now.get(k) != PIN0[k]]; new = [k for k in now if k not in PIN0]; return (not bad and not new), (f'{len(PIN0)} pre-existing files (incl. notebooks 07 and 08 outputs) unchanged; nothing new outside staging' if not bad and not new else f'CHANGED {bad[:3]} NEW {new[:3]}')
def e02():
    sell = {100: 1.0, 99: 2.0, 98: 1.0, 96: 5.0}; buy = {101: 1.0, 102: 2.0, 103: 1.0, 105: 5.0}
    cases = [(sell, 100, -1, 103, [100, 100], [0.4, 0.4], [0, 0]), (sell, 100, -1, 103, [100, 100], [1.0, 0.1], [0, 2]), (sell, 100, -1, 103, [99], [0.3], [1]), (sell, 100, -1, 103, [100, 99], [1.0, 0.5], [0, 0]), (sell, 100, -1, 103, [97], [0.1], [4]),
             (sell, 100, -1, 103, [101], [0.1], [5]), (sell, 100, -1, 101, [101], [0.1], [6]), (sell, 100, -1, 103, [100, 99, 98], [1.0, 2.0, 0.5], [0, 0, 0]), (sell, 100, -1, 103, [100 - 40000], [0.1], [7]), (sell, 100, -1, 103, [98, 98], [0.2, 0.2], [1, 1]), (sell, 100, -1, 103, [98], [5.0], [3]),
             (buy, 101, 1, 100, [101, 101], [0.4, 0.7], [0, 2]), (buy, 101, 1, 100, [102], [0.3], [1]), (buy, 101, 1, 100, [101, 102, 103], [1.0, 2.0, 0.4], [0, 0, 0]), (buy, 101, 1, 100, [104], [0.1], [4]), (buy, 101, 1, 98, [100], [0.1], [5]), (buy, 101, 1, 100, [100], [0.1], [6])]
    bad = []
    for i, (d, t, dr, o, p, a, exp) in enumerate(cases):
        got = walk_classify(d, t, dr, o, p, a)[0]
        if got != exp: bad.append((i, exp, got))
    return not bad, f'{len(cases)} hand-made interval cases (sells and buys; exact exhaustion, both tests violated, refill, skipped depth, undisplayed level, improved price, through opposite, walk cap) all classified as expected' if not bad else f'WRONG {bad[:3]}'
def e03():
    rnd = random.Random(12345); bad = 0; N = 4000
    for _ in range(N):
        dr = rnd.choice([-1, 1]); touch = 1000; dct = {}
        for k in range(0, 80):
            if k == 0 or rnd.random() < 0.7: dct[touch + dr * k] = round(rnd.uniform(0.01, 2.0), 3)
        opp = touch - dr * rnd.randint(1, 4); n = rnd.randint(1, 8); pr = [touch + dr * rnd.randint(-3, 60) for _ in range(n)]; am = [round(rnd.uniform(0.001, 1.5), 3) for _ in range(n)]
        if walk_classify(dct, touch, dr, opp, pr, am)[0] != brute(dct, touch, dr, opp, pr, am): bad += 1
    return bad == 0, f'walk_classify == independent brute-force implementation on {N:,} random books/trade sequences ({bad} disagreements)'
def e04():
    m_ok = st['messages'] == SRC_REPORT['coverage']['messages']; r_ok = st['rows'] == SRC_REPORT['book_scan']['rows']; c_ok = st['crossed'] == SRC_REPORT['crossed']['unpruned']['crossed_messages'] and st['locked'] == SRC_REPORT['crossed']['unpruned']['locked_messages']
    mm = re.search(r'unpruned: messages [\d,]+ .*?final levels \((\d+), (\d+)\)', SRC_SHARE); f_ok = bool(mm) and PS['final_levels'] == (int(mm.group(1)), int(mm.group(2)))
    return m_ok and r_ok and c_ok and f_ok, f"replay == notebook 07 record: messages {st['messages']:,} {m_ok}, rows {r_ok}, crossed/locked {c_ok}, final levels {PS['final_levels']} {f_ok}"
def e05():
    a = run_pass(max_rows=300_000, verbose=False); b = run_pass(max_rows=300_000, verbose=False); dig = lambda r: hashlib.sha256(r['cls'].tobytes() + r['dist'].tobytes() + r['Db'].tobytes() + r['Cp'].tobytes() + r['post_ok'].tobytes() + r['ML'].tobytes()).hexdigest()[:16]
    return dig(a) == dig(b) and a['stats']['messages'] == b['stats']['messages'], f"two 300k-row runs identical (digest {dig(a)}, {a['stats']['messages']:,} messages, {a['trades_evaluated']:,} trades evaluated)"
def e06():
    m = PS['age'] >= 0; a_ok = bool(np.all(PS['age'][m] > 0)); idx = np.searchsorted(MLn, tL, side='left') - 1; c_ok = bool(np.all(MLn[idx[idx >= 0]] < tL[idx >= 0])); return a_ok and c_ok, 'every trade was judged against a book interval that started strictly BEFORE it (age > 0; state timestamp < trade timestamp) - no look-ahead'
def e07():
    tot = int(np.bincount(cls, minlength=10).sum()); sums = {c: float(CLSTAB[c].sum()) for c in CLSTAB.columns if c.endswith('share')}; ok = tot == NT and all(abs(v_ - 1.0) < 1e-9 or (abs(v_) < 1e-12 and k_ == 'deep_Q_(>=2_ticks_deeper)_share' and NQ == 0) for k_, v_ in sums.items()); return ok, f'every one of {NT:,} trades has exactly one class; shares sum to 1: {sums}'
def e08():
    if not DIAG08.is_file(): return True, 'notebook 08 report not found -> cross-notebook reproduction skipped'
    rep = json.loads(DIAG08.read_text()); q08 = rep['categories']['Q_count']; thr08 = rep['categories']['table']['all_trades_count']['through_opposite']; ins08 = rep['categories']['table']['all_trades_count']['inside_spread']
    okq = int(NQ) == int(q08); okt = int((cls == 6).sum()) == int(thr08); oki = int((cls == 5).sum()) == int(ins08); return okq and okt and oki, f'pre-trade categories reproduce notebook 08: deep {NQ:,} vs {q08:,} {okq}; through-opposite {int((cls==6).sum()):,} vs {thr08:,} {okt}; inside-spread {int((cls==5).sum()):,} vs {ins08:,} {oki}'
def e09():
    if not HAVE_S25: return True, 'book_snapshot_25 not present -> best bid/ask re-check skipped'
    n_ = st['snap_checked']; r = sh(st['snap_best_equal'], n_); need = min(5000, len(SAMP) // 2); return n_ >= need and r >= 0.999, f'best bid & ask equal to book_snapshot_25 at {n_:,} aligned checkpoints: {r:.4%} (need >= 99.9% on >= {need:,}; same-provider consistency only)'
for nm, fn in (('E01 nothing pre-existing modified (read-only)', e01), ('E02 walk-the-book correct on hand-made cases', e02), ('E03 walk-the-book == brute-force implementation', e03), ('E04 replay reproduces notebook 07 numbers', e04), ('E05 replay deterministic', e05), ('E06 no look-ahead in the book state', e06), ('E07 classes partition all trades', e07), ('E08 pre-trade categories reproduce notebook 08', e08), ('E09 best bid/ask re-check vs book_snapshot_25', e09)): tcase('E', nm, fn)
def d01(): bad = [k for k, v_ in SHA_CHECK.items() if not v_['equal']]; return bool(SHA_CHECK) and not bad, ('SHA-256 of every saved raw file equals the value recorded by notebook 07' if not bad else f'MISMATCH {bad}')
def d02(): a = NT == SRC_REPORT['trades_scan']['rows']; b = st['rows'] == SRC_REPORT['book_scan']['rows']; return a and b, f"row counts equal notebook 07: trades {NT:,} {a}, book {st['rows']:,} {b}"
def d03(): return GRID['book_first_block_off_grid_share'] <= 1e-4 and GRID['trades_off_grid_share'] <= 1e-4, f'prices lie on the {TICK} tick grid: {GRID}'
def d04(): ok = bool(np.all(tpx > 0)) and bool(np.all(tam > 0)) and sh(int((tside == 2).sum()), NT) <= 0.01 and bool(np.all(np.isfinite(tpx))); return ok, f'trade price>0, amount>0, finite; unknown-side share {sh(int((tside == 2).sum()), NT):.4%}'
def d05(): dif = np.diff(tid); return bool(np.all(dif > 0)), f'trade IDs strictly increasing; gaps(>1): {int((dif > 1).sum())}'
def d06(): inside = float(np.mean((tL >= DAY0_US) & (tL < DAY1_US))); return inside >= 0.999, f'trade local_timestamp inside {DAY} UTC: {inside:.4%}'
def d07(): return bool(np.all(np.diff(MLn) > 0)) and len(MLn) > 1000, f'book message local timestamps strictly increasing ({len(MLn):,} messages)'
for nm, fn in (('D01 saved raw files match notebook 07 SHA-256', d01), ('D02 row counts match notebook 07', d02), ('D03 prices on tick grid', d03), ('D04 trade values sane', d04), ('D05 trade IDs increasing', d05), ('D06 trades inside the day', d06), ('D07 book messages strictly ordered', d07)): tcase('D', nm, fn)
E_OK = all(t[1] for t in TESTS['E']); D_OK = all(t[1] for t in TESTS['D'])
print(f"\nE (execution): {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])} | D (input integrity): {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])}")

def qlabel(x): return 'n/a' if x != x else 'LARGELY consistent with consuming the displayed book' if x >= 0.60 else 'PARTLY consistent' if x >= 0.30 else 'MOSTLY NOT consistent'
if not (E_OK and D_OK): REC = 'HOLD - an execution or integrity test failed; fix that first.'
elif W >= 0.70 and I_ <= 0.06: REC = 'PROCEED - ask for approval of the remaining free days (1st of each month in the study window). Use trades only as per-100ms-interval aggregates.'
elif W >= 0.40: REC = 'PROCEED FOR BOOK DATA ONLY - ask for approval of the remaining free days for the order-book stream, but do NOT match trades price-by-price to the book; use trades only as per-100ms-interval aggregates (signed volume, count, range).'
else: REC = 'HOLD - fewer than 40% of trades are consistent with the displayed book; do not download more until this is understood or raised with Tardis.'
print(REC)
def pct(x, d=2): return 'nan' if x != x else f'{x:.{d}%}'
unc = ['One symbol and one day.', 'A violation of either test is not a data error: the book is a 100 ms net-change feed, so cancellations, new orders and refills inside an interval are invisible by construction. The tests measure how much of the tape can be tied to the displayed book, nothing more.',
       'Trade order inside an interval follows the file (local arrival) order; trades of one interval that arrive out of true execution order would blur the reach/exhaustion tests.',
       'Book and trades come from the same provider; "consistent" is not independent validation. L2 data has no order identity: nothing here identifies orders or traders and nothing here detects manipulation.',
       'The 60% / 30% / 70% / 40% / 6% cut-offs are judgement calls fixed in advance, not statistical tests.', 'Publication of results is not addressed in the Tardis terms: ask Tardis in writing. Raw files, per-trade tables derived from them, and trained models must not be given to third parties.']
L = []; P = L.append
P('=' * 100); P(f'FINAL WALK-THE-BOOK SHARE BLOCK - Tardis.dev Binance Spot {SYMBOL} {DAY} (read-only follow-up to the audit)'); P('=' * 100)
P(f'run: {RUN_STARTED} | source (read-only): {SRC} | output: {OUT_FINAL} | network used: NO | downloads: NONE')
P('Question: at 100 ms resolution, are trades consistent with the displayed full-depth book being consumed? Reach test + exhaustion test per book interval and taker side. Not a validation of anything; no manipulation claims.')
P('\n[A] INPUTS'); [P(f"  {v_['file']}: sha256 equal to notebook 07 manifest: {v_['equal']}") for v_ in SHA_CHECK.values()]; P(f'  trades {NT:,} | book rows {st["rows"]:,} | tick {TICK} | usable trades {NV:,} | (interval,side) groups {st["groups"]:,}')
P('\n[B] REFERENCE REPLAY'); P(f"  messages {st['messages']:,} | crossed {st['crossed']} | locked {st['locked']} | resets {st['resets']} | final levels {PS['final_levels']} | best bid/ask equal to book_snapshot_25 at {st['snap_checked']:,} aligned checkpoints: {pct(sh(st['snap_best_equal'], st['snap_checked']), 4)} (same provider: consistency only)")
P('\n[C] TRADE CLASSES'); P(CLSTAB.assign(**{c: CLSTAB[c].map('{:.3%}'.format) for c in CLSTAB.columns if c.endswith('share')}).to_string())
P(f'  W (consistent, all usable) = {pct(W)} | I (better than touch or through opposite) = {pct(I_)} | deep trades Q = {NQ:,} ({pct(sh(NQ, NV))}) | Qc (deep and consistent) = {pct(Qc)} -> deep trades are {qlabel(Qc)} | deep trades at a price that is not a displayed level: {pct(Q_NOT_DISPLAYED)}')
P('\n[D] BY BURST SIZE (same-side trades in one book interval)'); P(BURST.assign(**{c: BURST[c].map('{:.2%}'.format) for c in BURST.columns if c not in ('burst_size_(same_side_trades_in_interval)', 'trades')}).to_string(index=False))
P('\n[E] SIZE OF VIOLATIONS'); P('  ' + json.dumps(MAG, default=jsonable))
P('\n[F] BOOK AFTER THE INTERVAL (touch moved to/through the deepest print?)'); P('  ' + json.dumps(POST, default=jsonable)); P('  volume-weighted class shares: ' + json.dumps(vol_w, default=jsonable))
P('\n[G] TIME OF DAY / VOLATILITY (share consistent)'); P(HOUR.assign(consistent_share=HOUR.consistent_share.map('{:.1%}'.format)).T.to_string()); P(VOL.assign(consistent_share=VOL.consistent_share.map('{:.1%}'.format)).to_string())
P('\n' + '-' * 100); P('[E] NOTEBOOK EXECUTION TESTS'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['E']]; P(f"  => E: {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])}")
P('[D] INPUT-DATA INTEGRITY TESTS'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['D']]; P(f"  => D: {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])}")
P('  Passing E and D means the code and the files are sound; it does NOT validate the book or the trades.')
REQ = {'stage_seconds': STAGES, 'peak_memory_gb': round(peak_mem_gb(), 2), 'replay_seconds': round(PS['seconds'], 1)}; P('\n[H] PROCESSING'); P('  ' + json.dumps(REQ, default=jsonable))
P('\nRECOMMENDATION (rule-based, frozen in advance; nothing is downloaded without your approval):'); P('  ' + REC)
P('\nUNCERTAINTIES'); [P('  - ' + u) for u in unc]
P('\nNothing was downloaded. No models, graphs or synthetic scenarios were created. Notebooks 07 and 08 outputs and all earlier artifacts were not modified. WAITING FOR YOUR APPROVAL before the next stage.'); P('=' * 100)
SHARE = '\n'.join(L)
STAGE.mkdir(exist_ok=True); (STAGE / 'reports').mkdir(exist_ok=True); REPORT['tests'] = TESTS; REPORT['recommendation'] = REC; REPORT['uncertainties'] = unc; REPORT['finished'] = UTC(); REPORT['processing'] = REQ
dump_json(REPORT, STAGE / 'reports' / 'walk_the_book_report.json'); (STAGE / 'reports' / 'share_block.txt').write_text(SHARE, encoding='utf-8')
CLSTAB.to_csv(STAGE / 'reports' / 'class_table.csv'); BURST.to_csv(STAGE / 'reports' / 'burst_table.csv', index=False); HOUR.to_csv(STAGE / 'reports' / 'hour_table.csv'); VOL.to_csv(STAGE / 'reports' / 'volatility_table.csv')
pd.DataFrame({'trade_id': tid, 'side_code(0buy,1sell,2other)': tside, 'class_code': cls, 'ticks_deeper_than_touch': np.clip(dist, -10 ** 6, 10 ** 6).astype(np.int32), 'same_side_trades_in_interval': gsz, 'touch_moved_to_or_through_deepest_print': pok}).to_parquet(STAGE / 'reports' / 'per_trade_walk_class.parquet', index=False)
MAN2 = {str(p.relative_to(STAGE)): sha256_file(p) for p in sorted(STAGE.rglob('*')) if p.is_file()}; dump_json(MAN2, STAGE / 'manifest.json')
okm = all(sha256_file(STAGE / k) == v_ for k, v_ in MAN2.items()); all_ok = E_OK and D_OK and okm
if all_ok: os.rename(STAGE, OUT_FINAL); print('\nSAVED ->', OUT_FINAL, '(results; the per-trade table is derived from licensed data - keep private)')
else: print('\n!! E or D tests FAILED (or manifest mismatch) -> staging kept at', STAGE, '(not renamed). Share this block anyway.')
print(SHARE); print(f'manifest verified: {okm} ({len(MAN2)} files)')

Writing tardis_body_walk.py


## 3 · Safety gate: do all 18 files exist, how big are they, is there room? (headers only, HEAD first then a GET whose body is never read — NOTHING is downloaded here)

In [ ]:
HEADERS = {'User-Agent': 'TradeSense-research-notebook/1.0 (final-year project)'}
TYPES = {'incremental_book_L2': True, 'trades': True, 'book_snapshot_25': False}
def url_for(sym, day, dt): y, m, d = day.split('-'); return f'{BASE_URL}/v1/{EXCHANGE}/{dt}/{y}/{m}/{d}/{sym}.csv.gz'
def probe(url):
    out = {'status': None, 'bytes': None, 'error': None}
    def size(h):
        try: return int(h['Content-Length']) if 'Content-Length' in h else None
        except ValueError: return None
    try:
        with requests.head(url, headers=HEADERS, allow_redirects=True, timeout=60) as r:       # HEAD has no body
            if r.status_code in (200, 206): out['status'] = r.status_code; out['bytes'] = size(r.headers); return out
    except Exception as e: out['error'] = 'HEAD ' + type(e).__name__ + ': ' + str(e)[:100]
    try:
        with requests.get(url, headers=HEADERS, stream=True, allow_redirects=True, timeout=60) as r:    # headers only: the body is never read; the response is closed on leaving the block
            out['status'] = r.status_code
            if r.status_code in (200, 206): out['bytes'] = size(r.headers)
    except Exception as e: out['error'] = (out['error'] + ' | ' if out['error'] else '') + 'GET ' + type(e).__name__ + ': ' + str(e)[:100]
    return out
TODO = [(s, d) for s, d in JOBS if not final_dir(s, d).exists()]
PR = {(s, d, dt): probe(url_for(s, d, dt)) for s, d in TODO for dt, req in TYPES.items() if req or INCLUDE_BOOK_SNAPSHOT_25}
tab = pd.DataFrame([{'symbol': s, 'day': d, 'type': dt, 'status': v['status'], 'MB': (v['bytes'] or 0) / 1e6, 'error': v['error']} for (s, d, dt), v in PR.items()]); display(tab)
def ok_probe(v): return v['status'] in (200, 206) and v['bytes'] is not None and v['bytes'] >= 1000
bad_sets = sorted({(s, d) for (s, d, dt), v in PR.items() if not ok_probe(v)}); RUN_JOBS = [j for j in TODO if j not in bad_sets]
if bad_sets:
    print('\n!! Not available (404 = no such file; 401/403 = free access does not apply; tiny file = Tardis\' empty answer):', bad_sets)
    if not ALLOW_PARTIAL: raise RuntimeError('STOPPED ON PURPOSE: some files are unavailable (see above). Nothing was downloaded. Paste this output to Claude; do NOT try other URLs.')
    print('ALLOW_PARTIAL=True -> continuing with:', RUN_JOBS)
sizes = {(s, d): sum((PR[(s, d, dt)]['bytes'] or 0) for dt in TYPES if (s, d, dt) in PR) / 1e9 for s, d in RUN_JOBS}; total_gb = sum(sizes.values())
free_gb = shutil.disk_usage(str(BASE)).free / 1e9; print(f'\nsets to download: {len(RUN_JOBS)} | total compressed {total_gb:.2f} GB | Drive free {free_gb:.1f} GB | per-set sizes (GB): { {f"{s} {d}": round(v, 3) for (s, d), v in sizes.items()} }')
if any(v > MAX_GB_PER_SET for v in sizes.values()): raise RuntimeError(f'STOPPED ON PURPOSE: a set exceeds MAX_GB_PER_SET={MAX_GB_PER_SET}.')
if total_gb * 1.5 + 5 > free_gb: raise RuntimeError(f'STOPPED ON PURPOSE: not enough free space (need about {total_gb*1.5+5:.1f} GB).')
if not RUN_JOBS: print('Nothing to do: every set already exists.')
if not I_HAVE_READ_THE_TERMS:
    print('\n' + '=' * 90 + '\nGATE: all files are available and there is space. NOTHING HAS BEEN DOWNLOADED.\nTo continue: read the Tardis Terms of Service, set I_HAVE_READ_THE_TERMS = True in cell 1, then Run all.\n' + '=' * 90)
    raise RuntimeError('STOPPED ON PURPOSE at the safety gate (this is expected the first time).')
print('Gate passed.')

,symbol,day,type,status,MB,error
0,ETHUSDT,2024-09-01,incremental_book_L2,200,133.080514,None
1,ETHUSDT,2024-09-01,trades,200,11.697856,None
2,ETHUSDT,2024-09-01,book_snapshot_25,200,46.407201,None
3,BTCUSDT,2024-10-01,incremental_book_L2,200,188.273685,None
4,BTCUSDT,2024-10-01,trades,200,42.818702,None
5,BTCUSDT,2024-10-01,book_snapshot_25,200,50.485587,None
6,ETHUSDT,2024-10-01,incremental_book_L2,200,173.750354,None
7,ETHUSDT,2024-10-01,trades,200,35.408245,None
8,ETHUSDT,2024-10-01,book_snapshot_25,200,56.709311,None



sets to download: 3 | total compressed 0.74 GB | Drive free 88.9 GB | per-set sizes (GB): {'ETHUSDT 2024-09-01': 0.191, 'BTCUSDT 2024-10-01': 0.282, 'ETHUSDT 2024-10-01': 0.266}
Gate passed.


## 4 · Download and audit, one set after the other (this is the long part)

For each set: download → integrity → schema → timestamps/coverage → full-depth replay against `book_snapshot_25` → E/D tests → share block → folder saved only if E and D pass. A failure in one set does not stop the others; the status table at the end says what happened.

In [ ]:
STATUS = {}
def run_body(fname, extra):
    ns = {'__name__': '__main__', 'display': display, **extra}
    exec(compile(Path(fname).read_text(), fname, 'exec'), ns); return ns
for sym, day in RUN_JOBS:
    key = f'{sym} {day}'; print('\n' + '#' * 100 + f'\n# AUDIT {key}\n' + '#' * 100)
    if final_dir(sym, day).exists(): STATUS[key] = 'already saved (skipped)'; continue
    t_job = time.time()
    try:
        ns = run_body('tardis_body_audit.py', {'JOB_SYMBOL': sym, 'JOB_DAY': day, 'DRIVER_I_HAVE_READ_THE_TERMS': I_HAVE_READ_THE_TERMS, 'DRIVER_INCLUDE_S25': INCLUDE_BOOK_SNAPSHOT_25, 'DRIVER_MAX_GB': MAX_GB_PER_SET})
        STATUS[key] = ('audit saved' if final_dir(sym, day).exists() else 'audit finished but E/D tests failed -> staging kept (hidden .tardis_stage_* folder in Drive)') + f' | {time.time()-t_job:.0f}s'
    except Exception as e:
        STATUS[key] = f'FAILED: {type(e).__name__}: {str(e)[:200]}'; print(traceback.format_exc()[-1500:])
    ns = None; gc.collect()
print('\nAUDIT STATUS:'); [print('  ', k, '->', v) for k, v in STATUS.items()]


####################################################################################################
# AUDIT ETHUSDT 2024-09-01
####################################################################################################
fingerprinted 885 existing files (they will be re-checked at the end)
[fingerprint] 26.4 s | peak memory so far 0.19 GB
incremental_book_L2 -> https://datasets.tardis.dev/v1/binance/incremental_book_L2/2024/09/01/ETHUSDT.csv.gz
trades -> https://datasets.tardis.dev/v1/binance/trades/2024/09/01/ETHUSDT.csv.gz
book_snapshot_25 -> https://datasets.tardis.dev/v1/binance/book_snapshot_25/2024/09/01/ETHUSDT.csv.gz
Drive/target volume: total 115.7 GB | free 88.9 GB   ||  local disk free 93.6 GB


,status,bytes,content_type,last_modified,error,body
incremental_book_L2,200,133080514,text/csv,None,None,None
trades,200,11697856,text/csv,None,None,None
book_snapshot_25,200,46407201,text/csv,None,None,None



Total compressed size of the files above: 0.19 GB (files with unknown size: 0)
Gate passed. Proceeding to download.
downloading incremental_book_L2
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 8 MB (6%) 0s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 17 MB (13%) 0s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 25 MB (19%) 0s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 34 MB (25%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 42 MB (32%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 50 MB (38%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 59 MB (44%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 67 MB (50%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 75 MB (57%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 84 MB (63%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.csv.gz: 92 MB (69%) 1s
   binance_incremental_book_L2_2024-09-01_ETHUSDT.

,start_row,local_ts,rows,bid_rows,ask_rows,time_utc
0,0,1725148800758245,2010,1009,1001,2024-09-01T00:00:00.758245+00:00


rows per UTC hour: {0: 552161, 1: 641879, 2: 1072490, 3: 475413, 4: 581292, 5: 1109966, 6: 607078, 7: 508712, 8: 464384, 9: 506130, 10: 723256, 11: 612192, 12: 619786, 13: 956426, 14: 1658354, 15: 1169588, 16: 911737, 17: 823169, 18: 1062183, 19: 913349, 20: 648034, 21: 549198, 22: 2126471, 23: 1556670}
[scan_trades] 2.7 s | peak memory so far 1.58 GB
{
 "local_outside_day_rows": 0,
 "amount_min": 0.0001,
 "rows": 1357443,
 "timestamp_min": "2024-09-01T00:00:00.790000+00:00",
 "timestamp_max": "2024-09-01T23:59:59.915000+00:00",
 "local_min": "2024-09-01T00:00:00.793098+00:00",
 "local_max": "2024-09-01T23:59:59.918051+00:00",
 "local_inversions": 0,
 "sides": {
  "sell": 715445,
  "buy": 641998
 },
 "price_range": [
  2400.0,
  2516.28
 ],
 "amount_range": [
  0.0001,
  380.6381
 ],
 "id_numeric": true,
 "id_gaps(>1)": 0,
 "id_dup_or_backwards": 0,
 "local_minus_exchange_us_median": 3146.0,
 "local_minus_exchange_us_p99": 28468.719999999885,
 "gap_counts": {
  ">1s": 26290,
  ">10s": 

...

rows per UTC hour: {0: 1366539, 1: 830467, 2: 612043, 3: 644381, 4: 709445, 5: 452837, 6: 578936, 7: 581615, 8: 499294, 9: 502694, 10: 460771, 11: 423182, 12: 553928, 13: 1673593, 14: 2391660, 15: 2438587, 16: 2230106, 17: 2692668, 18: 1662930, 19: 1574952, 20: 2619234, 21: 1087185, 22: 1069850, 23: 913728}
[scan_trades] 4.6 s | peak memory so far 2.39 GB
{
 "local_outside_day_rows": 0,
 "amount_min": 0.0001,
 "rows": 4525441,
 "timestamp_min": "2024-10-01T00:00:00.344000+00:00",
 "timestamp_max": "2024-10-01T23:59:59.443000+00:00",
 "local_min": "2024-10-01T00:00:00.347745+00:00",
 "local_max": "2024-10-01T23:59:59.445629+00:00",
 "local_inversions": 0,
 "sides": {
  "buy": 2166678,
  "sell": 2358763
 },
 "price_range": [
  2414.0,
  2659.0
 ],
 "amount_range": [
  0.0001,
  283.0189
 ],
 "id_numeric": true,
 "id_gaps(>1)": 1,
 "id_dup_or_backwards": 0,
 "local_minus_exchange_us_median": 3334.0,
 "local_minus_exchange_us_p99": 61255.54999999984,
 "gap_counts": {
  ">1s": 17807,
  ">10

,messages_checked,crossed_messages,locked_messages,crossed_share,locked_share,crossed_plus_locked_share,max_consecutive_crossed
unpruned,845676.0,0.0,0.0,0.0,0.0,0.0,0.0


,unpruned
snapshot25_rows_total,753604
snapshot25_rows_sampled,20368
distinct_local_timestamps_sampled,20368
matched_to_a_completed_message,20368
unmatched_no_message_with_equal_local_timestamp,0
excluded_book_crossed_or_locked,0
excluded_book_empty,0
ALIGNED_CHECKPOINTS,20368
top5_price_agreement_(levels),1.0
top5_quantity_agreement_(price AND qty equal),1.0


{
 "messages": 845676,
 "minutes_with_a_message_of_1440": 1440,
 "coverage_of_day_by_minutes": 1.0,
 "first_message_utc": "2024-10-01T00:00:01.133064+00:00",
 "last_message_utc": "2024-10-01T23:59:59.916995+00:00",
 "messages_per_second_mean": 9.788054432286637,
 "gaps_between_messages_over": {
  ">1s": 0,
  ">10s": 0,
  ">60s": 0,
  ">300s": 0
 },
 "largest_message_gaps_s": [
  0.9,
  0.9,
  0.8,
  0.8,
  0.7,
  0.7,
  0.7,
  0.7
 ]
}
{
 "unpruned": {
  "spread_bps_median": 0.03823748152561255,
  "levels_start": [
   1004,
   1000
  ],
  "levels_end": [
   11390,
   18560
  ],
  "levels_max": [
   12198,
   18874
  ],
  "far_levels_max_(>5pct_from_mid)": 20923,
  "depth10bps_median_btc": 1419.7762000000005,
  "depth25bps_median_btc": 3719.8476999999953,
  "samples": 2875
 }
}
{
 "download_total_mb": 265.9,
 "uncompressed_total_gb": 3.04,
 "rows_incremental_book_L2": 28570625,
 "rows_trades": 4525441,
 "peak_memory_gb": 2.41,
 "stage_seconds": {
  "fingerprint": {
   "seconds": 1.2,
  

[None, None, None]

## 5 · Walk-the-book test on each saved set (read-only; same test as notebook 09)

In [ ]:
WSTATUS = {}
if RUN_WALK_THE_BOOK:
    for sym, day in JOBS:
        key = f'{sym} {day}'
        if not final_dir(sym, day).exists(): WSTATUS[key] = 'no saved audit -> skipped'; continue
        if walk_dir(sym, day).exists(): WSTATUS[key] = 'already done (skipped)'; continue
        print('\n' + '#' * 100 + f'\n# WALK-THE-BOOK {key}\n' + '#' * 100); t_job = time.time()
        try:
            ns = run_body('tardis_body_walk.py', {'JOB_SYMBOL': sym, 'JOB_DAY': day}); WSTATUS[key] = ('saved' if walk_dir(sym, day).exists() else 'finished but E/D tests failed -> staging kept') + f' | {time.time()-t_job:.0f}s'
        except Exception as e:
            WSTATUS[key] = f'FAILED: {type(e).__name__}: {str(e)[:200]}'; print(traceback.format_exc()[-1500:])
        ns = None; gc.collect()
print('\nWALK-THE-BOOK STATUS:'); [print('  ', k, '->', v) for k, v in WSTATUS.items()]


####################################################################################################
# WALK-THE-BOOK BTCUSDT 2024-08-01
####################################################################################################
fingerprinted 909 existing files (including the 07 output folder); they will be re-checked at the end
[fingerprint] 1.0 s | peak memory so far 2.41 GB
source folder: /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-08-01_v1
book_snapshot_25 present: True
incremental_book_L2 sha256 equal to manifest: True
trades sha256 equal to manifest: True
book_snapshot_25 sha256 equal to manifest: True
[sha256_recheck] 6.0 s | peak memory so far 2.41 GB
[load_trades] 2.6 s | peak memory so far 2.41 GB
trades loaded: 2,397,256 | buy 1,103,698 sell 1,293,558 other 0
trades sorted by local ts: True | by exchange ts: True
tick size inferred from the data: 0.01
{'book_first_block_off_grid_share': 0.0, 'trades_off_grid_share': 0.0}
[load_snap

,all_count,all_share,sell_taker_count,sell_taker_share,buy_taker_count,buy_taker_share,deep_Q_(>=2_ticks_deeper)_count,deep_Q_(>=2_ticks_deeper)_share,not_deep_count,not_deep_share
consistent,1381035,57.609%,784058,60.613%,596977,54.089%,154193,16.605%,1226842,83.536%
reach_violated(deeper_than_volume_explains),645543,26.929%,318567,24.627%,326976,29.626%,590083,63.545%,55460,3.776%
exhaustion_violated(more_than_displayed),154910,6.462%,75315,5.822%,79595,7.212%,90356,9.730%,64554,4.396%
both_violated,259,0.011%,156,0.012%,103,0.009%,251,0.027%,8,0.001%
price_not_displayed_level,95949,4.002%,50073,3.871%,45876,4.157%,93721,10.093%,2228,0.152%
better_than_touch_inside_spread,17118,0.714%,6687,0.517%,10431,0.945%,0,0.000%,17118,1.166%
at_or_through_opposite_touch,102426,4.273%,58696,4.538%,43730,3.962%,0,0.000%,102426,6.974%
beyond_walk_cap,0,0.000%,0,0.000%,0,0.000%,0,0.000%,0,0.000%


usable trades 2,397,240 | W (consistent, all) = 57.61% | I (better than touch / through opposite) = 4.99% | deep trades Q = 928,604 (38.74%) | Qc (deep trades that are consistent) = 16.60%


,burst_size_(same_side_trades_in_interval),trades,share_of_usable,P(deep),consistent,reach_violated,exhaustion_violated,both,not_displayed,better_than_touch_or_through
0,1 trade,224026,9.35%,0.34%,95.99%,0.42%,0.00%,0.00%,0.02%,3.57%
1,2-3 trades,140332,5.85%,2.68%,92.23%,2.23%,0.15%,0.00%,0.26%,5.13%
2,4-10 trades,241188,10.06%,21.55%,75.45%,16.20%,2.55%,0.01%,2.27%,3.52%
3,>10 trades,1791694,74.74%,48.67%,47.70%,33.62%,8.29%,0.01%,5.03%,5.35%


{
 "magnitudes": {
  "reach_violated_trades": 645802,
  "shortfall_BTC_quantiles(50,75,90,99)": [
   0.5293100000000001,
   1.2318775,
   2.6107399999999994,
   12.127425599999988
  ],
  "shortfall_as_share_of_better_depth_quantiles(50,75,90,99)": [
   0.7571215715910655,
   0.9336243030949678,
   0.9865627294836693,
   1.0
  ],
  "exhaustion_violated_trades": 155169,
  "excess_BTC_quantiles(50,75,90,99)": [
   0.010280000000000289,
   0.0941599999999998,
   0.4526100000000014,
   5.961276400000068
  ],
  "median_trade_amount_BTC": 0.00065,
  "interval_volume_over_touch_qty_quantiles(50,75,90,99)_for_deep_trades": [
   1.635143570536829,
   5.554425064599483,
   31.961967213114757,
   8706.818181818182
  ]
 },
 "post_interval_touch": {
  "consistent": {
   "trades_with_post_state": 1381035,
   "touch_moved_to_or_through_deepest_print": 0.9696119214936624
  },
  "reach_violated": {
   "trades_with_post_state": 645543,
   "touch_moved_to_or_through_deepest_print": 0.8975730509044324
  },

...

{
 "magnitudes": {
  "reach_violated_trades": 1152470,
  "shortfall_BTC_quantiles(50,75,90,99)": [
   7.8862000000000005,
   17.109874999999995,
   36.18114999999998,
   170.49778200000037
  ],
  "shortfall_as_share_of_better_depth_quantiles(50,75,90,99)": [
   0.8845622839198035,
   0.9831188112985382,
   0.9954923057214462,
   0.9992376833481426
  ],
  "exhaustion_violated_trades": 1257101,
  "excess_BTC_quantiles(50,75,90,99)": [
   0.027899999999999994,
   0.05260000000000009,
   0.17649999999999988,
   8.115100000000005
  ],
  "median_trade_amount_BTC": 0.0021,
  "interval_volume_over_touch_qty_quantiles(50,75,90,99)_for_deep_trades": [
   1.805770584095707,
   10.29215367164768,
   102.51851851851853,
   4852.466666666666
  ]
 },
 "post_interval_touch": {
  "consistent": {
   "trades_with_post_state": 1274836,
   "touch_moved_to_or_through_deepest_print": 0.9596732442447499
  },
  "reach_violated": {
   "trades_with_post_state": 1151779,
   "touch_moved_to_or_through_deepest_prin

,consistent_share,trades
h,,
0,25.80%,196802
1,30.91%,95290
2,33.32%,54090
3,32.55%,88300
4,38.84%,99066
5,39.87%,55137
6,38.63%,87150
7,37.69%,61155
8,41.88%,51096


,consistent_share,trades
vol_quintile,,
Q1 calmest,35.41%,905086
Q2,28.06%,905085
Q3,26.47%,905086
Q4,26.00%,905085
Q5 most volatile,24.91%,905086


PASS E01 nothing pre-existing modified (read-only) | 949 pre-existing files (incl. notebooks 07 and 08 outputs) unchanged; nothing new outside staging
PASS E02 walk-the-book correct on hand-made cases | 17 hand-made interval cases (sells and buys; exact exhaustion, both tests violated, refill, skipped depth, undisplayed level, improved price, through opposite, walk cap) all classified as expected
PASS E03 walk-the-book == brute-force implementation | walk_classify == independent brute-force implementation on 4,000 random books/trade sequences (0 disagreements)
PASS E04 replay reproduces notebook 07 numbers | replay == notebook 07 record: messages 845,676 True, rows True, crossed/locked True, final levels (11349, 18595) True
PASS E05 replay deterministic | two 300k-row runs identical (digest 279a6e45c4f79982, 7,711 messages, 46,677 trades evaluated)
PASS E06 no look-ahead in the book state | every trade was judged against a book interval that started strictly BEFORE it (age > 0; state t

[None, None, None, None, None, None]

## 6 · FINAL MULTI-DAY SHARE BLOCK (copy everything between the lines and paste it back to Claude)

One row per set: the six earlier sets and the six new ones. Everything is read back from the saved reports.

In [ ]:
def jload(p):
    try: return json.loads(Path(p).read_text())
    except Exception: return None
ALL_SETS = list(PREV_SETS) + list(JOBS); ROWS = []
for sym, day in ALL_SETS:
    a = jload(final_dir(sym, day) / 'reports' / 'audit_report.json'); w = jload(walk_dir(sym, day) / 'reports' / 'walk_the_book_report.json'); r = {'symbol': sym, 'day': day, 'audit_saved': a is not None, 'walk_done': w is not None}
    if a:
        cm = (a.get('cmp_snapshot25') or {}).get('unpruned', {}); cr = (a.get('crossed') or {}).get('unpruned', {}); t = a.get('tests', {})
        r.update({'book_rows': a['book_scan']['rows'], 'trades': a['trades_scan']['rows'], 'messages': (a.get('coverage') or {}).get('messages'), 'minutes_covered': (a.get('coverage') or {}).get('coverage_of_day_by_minutes'), 'crossed+locked': cr.get('crossed_plus_locked_share'),
                  'aligned_checkpoints': cm.get('ALIGNED_CHECKPOINTS'), 'top5_price': cm.get('top5_price_agreement_(levels)'), 'top25_price': cm.get('top25_price_agreement_(levels)'), 'top25_qty': cm.get('top25_quantity_agreement_(price AND qty equal)'), 'spread_equal': cm.get('spread_exactly_equal'),
                  'E_pass': f"{sum(x[1] for x in t.get('E', []))}/{len(t.get('E', []))}", 'D_pass': f"{sum(x[1] for x in t.get('D', []))}/{len(t.get('D', []))}", 'R_verdict': (a.get('reconstruction_verdict') or {}).get('unpruned', 'n/a')[:48], 'download_MB': (a.get('requirements') or {}).get('download_total_mb'), 'replay_s': (a.get('requirements') or {}).get('replay_seconds_both_variants')})
    if w:
        c = w['classes']; r.update({'W_consistent': c['W_consistent_all'], 'I_better_or_through': c['I_better_or_through'], 'Qc_deep_consistent': c['Qc_deep_consistent'], 'walk_E': f"{sum(x[1] for x in w['tests']['E'])}/{len(w['tests']['E'])}", 'walk_D': f"{sum(x[1] for x in w['tests']['D'])}/{len(w['tests']['D'])}", 'walk_rule': w.get('recommendation', '')[:34]})
    ROWS.append(r)
SUMM = pd.DataFrame(ROWS); display(SUMM)
done = [r for r in ROWS if r['audit_saved']]; ok_all = len(done) == len(ALL_SETS) and all(r['walk_done'] for r in ROWS) and all(r['E_pass'].split('/')[0] == r['E_pass'].split('/')[1] and r['D_pass'].split('/')[0] == r['D_pass'].split('/')[1] and r['R_verdict'].startswith('CONSISTENT') for r in done)
Wv = [r['W_consistent'] for r in ROWS if 'W_consistent' in r]
L = []; P = L.append
P('=' * 100); P('FINAL MULTI-DAY SHARE BLOCK (Stage B2) - Tardis.dev Binance Spot, free first-of-month days: 6 earlier sets (May-July 2024) + 6 new test-month sets (Aug-Oct 2024)'); P('=' * 100)
P(f'sets in this table: {len(ALL_SETS)} ({len(PREV_SETS)} earlier + {len(JOBS)} new Aug-Oct) | audits saved: {len(done)} | gate flag: {I_HAVE_READ_THE_TERMS}')
P('Audit status this run: ' + json.dumps(STATUS)); P('Walk-the-book status this run: ' + json.dumps(WSTATUS))
P('\n' + SUMM.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
P('\nREADING THE TABLE')
P('  E/D = execution and file-integrity tests (code and files are sound; says nothing about the book). R_verdict = frozen criteria R01-R05 on the FULL-DEPTH book vs the provider\'s own book_snapshot_25: consistency only, NOT independent validation.')
P('  W = share of trades exactly consistent with consuming the displayed 100 ms book (notebook 09); I = share priced better than the touch or through the opposite touch; Qc = same for the deep trades.')
if Wv: P(f'  W across sets: min {min(Wv):.1%}, max {max(Wv):.1%} (first set was 57.5%). Differences are descriptive; with only a handful of days per symbol no stability claim is possible, and these are audit statistics, not model-selection evidence.')
P(f'\nCONDITION FOR MOVING ON (frozen): every saved set has all E/D tests passed AND reconstruction verdict CONSISTENT -> {"MET" if ok_all else "NOT MET (see table)"}; sets without a saved audit or walk-the-book result: {[f"{r[chr(115)+chr(121)+chr(109)+chr(98)+chr(111)+chr(108)]} {r[chr(100)+chr(97)+chr(121)]}" for r in ROWS if not (r["audit_saved"] and r["walk_done"])]}; failures reported this run: {[k for k, v in {**STATUS, **WSTATUS}.items() if "FAILED" in v or "failed" in v]}')
P('If MET: the next step is a Stage C proposal (read-only causal graph-feature build with look-ahead tests and measured memory/speed) under TGNN Protocol Design v2: train May+June, validate July, test Aug-Oct 2024, both symbols. If NOT MET or any Aug-Oct set failed: the protocol fallback split applies and we discuss first. Nothing is built or trained without your approval.')
P('\nLICENCE REMINDER: internal research/educational use; no redistribution of raw data, derived per-trade tables, or trained models; publication of results is not addressed in the terms -> ask Tardis in writing.')
P('Nothing beyond the sets listed above was downloaded. No synthetic events, graphs or models were created. Existing research artifacts were not modified. WAITING FOR YOUR APPROVAL before the next stage.'); P('=' * 100)
SHARE = '\n'.join(L)
k = 1
while (BASE / f'TGNN_Tardis_MultiDay_Summary_v{k}').exists(): k += 1
OUTS = BASE / f'TGNN_Tardis_MultiDay_Summary_v{k}'; OUTS.mkdir(); SUMM.to_csv(OUTS / 'multiday_summary.csv', index=False); (OUTS / 'share_block.txt').write_text(SHARE, encoding='utf-8')
print(SHARE); print('\nsummary saved ->', OUTS)

,symbol,day,audit_saved,walk_done,book_rows,trades,messages,minutes_covered,crossed+locked,aligned_checkpoints,top5_price,top25_price,top25_qty,spread_equal,E_pass,D_pass,R_verdict,download_MB,replay_s,W_consistent,I_better_or_through,Qc_deep_consistent,walk_E,walk_D,walk_rule
0,BTCUSDT,2024-05-01,True,True,31005311,2401094,861138,1.0,0.0,20259,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,271.5,442.6,0.575123,0.044505,0.189618,9/9,7/7,PROCEED FOR BOOK DATA ONLY - ask f
1,ETHUSDT,2024-05-01,True,True,25576675,1365039,850130,1.0,0.0,20267,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,230.1,250.8,0.546792,0.049685,0.199970,9/9,7/7,PROCEED FOR BOOK DATA ONLY - ask f
2,BTCUSDT,2024-06-01,True,True,6771693,638497,843621,1.0,0.0,20683,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,69.6,132.6,0.871101,0.016060,0.293053,9/9,7/7,PROCEED - ask for approval of the
3,ETHUSDT,2024-06-01,True,True,7512492,399381,780373,1.0,0.0,20070,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,82.3,161.9,0.771382,0.045399,0.254347,9/9,7/7,PROCEED - ask for approval of the
4,BTCUSDT,2024-07-01,True,True,14765722,1461507,841616,1.0,0.0,20432,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,141.4,239.5,0.638331,0.029392,0.124490,9/9,7/7,PROCEED FOR BOOK DATA ONLY - ask f
5,ETHUSDT,2024-07-01,True,True,10348946,854102,773787,1.0,0.0,20370,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,110.2,185.2,0.659765,0.021314,0.162785,9/9,7/7,PROCEED FOR BOOK DATA ONLY - ask f
6,BTCUSDT,2024-08-01,True,True,41604072,2397256,860802,1.0,0.0,20349,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,308.8,407.4,0.576094,0.049867,0.166048,9/9,7/7,PROCEED FOR BOOK DATA ONLY - ask f
7,ETHUSDT,2024-08-01,True,True,36138289,1835455,859024,1.0,0.0,20358,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,287.8,279.3,0.450143,0.037116,0.128350,9/9,7/7,PROCEED FOR BOOK DATA ONLY - ask f
8,BTCUSDT,2024-09-01,True,True,24717509,2153180,856637,1.0,0.0,20371,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,207.3,319.7,0.385472,0.021441,0.132503,9/9,7/7,HOLD - fewer than 40% of trades ar
9,ETHUSDT,2024-09-01,True,True,20849918,1357443,843466,1.0,0.0,20284,1.0,1.0,1.0,1.0,6/6,10/10,CONSISTENT with the provider's own book_snapshot,191.2,146.1,0.354153,0.022680,0.083802,9/9,7/7,HOLD - fewer than 40% of trades ar


FINAL MULTI-DAY SHARE BLOCK (Stage B2) - Tardis.dev Binance Spot, free first-of-month days: 6 earlier sets (May-July 2024) + 6 new test-month sets (Aug-Oct 2024)
sets in this table: 12 (6 earlier + 6 new Aug-Oct) | audits saved: 12 | gate flag: True
Audit status this run: {"ETHUSDT 2024-09-01": "audit saved | 241s", "BTCUSDT 2024-10-01": "audit saved | 379s", "ETHUSDT 2024-10-01": "audit saved | 270s"}
Walk-the-book status this run: {"BTCUSDT 2024-08-01": "saved | 152s", "ETHUSDT 2024-08-01": "saved | 108s", "BTCUSDT 2024-09-01": "saved | 94s", "ETHUSDT 2024-09-01": "saved | 73s", "BTCUSDT 2024-10-01": "saved | 134s", "ETHUSDT 2024-10-01": "saved | 103s"}

 symbol        day  audit_saved  walk_done  book_rows  trades  messages  minutes_covered  crossed+locked  aligned_checkpoints  top5_price  top25_price  top25_qty  spread_equal E_pass D_pass                                        R_verdict  download_MB  replay_s  W_consistent  I_better_or_through  Qc_deep_consistent walk_E walk_D     